In [6]:
# ============================================================
# UNIFIED IB / ORB CURVE ANALYSIS  — v3 (+ Session Context & Signal Scoring)
# ============================================================
#
# Investigates the full auction-theory chain for NQ futures:
#
#   Asia/London context
#       → orb_close_half  (5-min and 15-min)
#           → orb_position_in_ib
#               → ib_extreme_first
#                   → ib_close_half
#                       → post-IB direction (10:30am → 4:00pm)
#
# SECTIONS
# --------
#   1A. ORB-5  Internal Structure   — where does the 5-min ORB close within its range?
#   1B. ORB-15 Internal Structure   — where does the 15-min ORB (Q1) close within its range?
#   2A. ORB-5  Position Within IB   — where does the 5-min ORB sit in the full IB range?
#   2B. ORB-15 Position Within IB   — where does the 15-min ORB sit in the full IB range?
#   2C. ORB → IB Extreme First      — can the ORB predict which IB extreme forms first?
#   3.  IB Quarter Structure        — which quarter sets IB high/low? curve patterns?
#   4A. ORB-5  Curve Analysis       — Q2 breaks ORB-5 range  → curve or continuation?
#   4B. ORB-15 Curve Analysis       — Q3 breaks ORB-15 range → curve or continuation?
#   5.  Q1 Curve Analysis           — Q2 breaks Q1 range     → curve or continuation?
#   6.  Chain Validation            — do orb_close_half / ib_extreme_first / ib_close_half agree?
#   7.  Post-IB Session Outcomes    — RTH direction conditioned on every pattern above
#   8.  Full Confluence Summary     — ranked edge table, all patterns
#   9.  Reverse Conditional         — given first IB break direction, what was ORB position?
#   10. Session Context             — Asia/London direction and positional relationship
#   11. Signal Scoring System       — 6-signal majority vote, trade type, ranked edge table
#
# ============================================================

import pandas as pd
import numpy as np
import warnings
from collections import defaultdict
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION — edit these to match your environment
# ============================================================

DATA_DIR      = r"C:\Users\tammm\NQ and ES analysis"
RAW_DATA_FILE = "nq_15y_1min.csv"
OUTPUT_DIR    = r"C:\Users\tammm\NQ and ES analysis"
START_DATE    = "2013-01-01"
END_DATE      = "2025-12-31"

# Minimum ORB range as a fraction of previous day's range.
# ORB close-half signal is ignored when ORB is too tight to be meaningful.
MIN_ORB_RANGE_FRACTION = 0.03   # 3% of prev day range

# Minimum signal agreement to output a directional bias (out of 6 signals)
MIN_SIGNAL_AGREEMENT = 4

# Minimum sample size to display a result
MIN_DISPLAY_SAMPLE = 30

import os
RAW_DATA_PATH    = os.path.join(DATA_DIR, RAW_DATA_FILE)
DATASET_CSV_PATH = os.path.join(OUTPUT_DIR, "ib_opportunity_dataset.csv")
EDGE_CSV_PATH    = os.path.join(OUTPUT_DIR, "ib_opportunity_edge_table.csv")


# ============================================================
# HELPERS
# ============================================================

def safe_pct(numerator, denominator):
    return (numerator / denominator * 100) if denominator > 0 else 0.0


def close_half(close_price, range_high, range_low):
    """Return 'upper' or 'lower' depending on which half close_price sits in."""
    if range_high == range_low:
        return None
    midpoint = (range_high + range_low) / 2
    return 'upper' if close_price >= midpoint else 'lower'


def position_in_range(price, range_high, range_low):
    """Return 0-1 score of where price sits in [range_low, range_high]. 0=bottom, 1=top."""
    if range_high == range_low:
        return None
    return (price - range_low) / (range_high - range_low)


def position_bucket(score):
    """Bucket a 0-1 position score into thirds."""
    if score is None:
        return None
    if score <= 1/3:
        return 'bottom_third'
    elif score <= 2/3:
        return 'middle_third'
    else:
        return 'top_third'


def session_direction(open_price, close_price):
    if open_price == close_price:
        return None
    return 'bullish' if close_price > open_price else 'bearish'


def print_header(text, width=90):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=90):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar_len = int(pct / 2)
    bar = "█" * bar_len
    print(f"{' '*indent}{label:<55} {count:5d}  ({pct:5.1f}%)  {bar}")


def print_outcome_row(label, bull, bear, indent=4):
    total = bull + bear
    if total < MIN_DISPLAY_SAMPLE:
        print(f"{' '*indent}{label:<55} n={total:4d}  (below min sample)")
        return
    bull_pct = safe_pct(bull, total)
    bear_pct = safe_pct(bear, total)
    edge = abs(bull_pct - 50)
    bias = "BULL" if bull_pct > 50 else "BEAR"
    print(f"{' '*indent}{label:<55} n={total:4d}  "
          f"Bull:{bull_pct:5.1f}%  Bear:{bear_pct:5.1f}%  "
          f"Edge:{edge:4.1f}%  [{bias}]")


# ============================================================
# LOAD DATA
# ============================================================

print_header("UNIFIED IB / ORB CURVE ANALYSIS  (5-min + 15-min ORB)")
print(f"\n  Loading data from: {RAW_DATA_PATH}")

df = pd.read_csv(RAW_DATA_PATH)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df = df.loc[START_DATE:END_DATE]

df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

print(f"  Loaded {len(df):,} bars | {len(dates)} trading days ({START_DATE} → {END_DATE})")


# ============================================================
# SESSION PATTERN LOOKUP TABLES & SIGNAL SCORING
# ============================================================
# Must be defined before the main loop which calls these functions.

SESSION_PATTERN_BIAS = {
    'bull_bull' : 'bullish',
    'bull_bear' : 'neutral',
    'bear_bull' : 'neutral',
    'bear_bear' : 'bearish',
}

ASIA_LONDON_BIAS = {
    'london_partial_up'   : 'bullish',
    'london_partial_down' : 'bearish',
    'london_breaks_both'  : 'neutral',
    'london_engulfs_asia' : 'neutral',
    'asia_engulfs_london' : 'neutral',
    'overlapping'         : 'neutral',
}

def score_session_pattern(pattern):
    return SESSION_PATTERN_BIAS.get(pattern, 'neutral')

def score_asia_london_position(pos):
    return ASIA_LONDON_BIAS.get(pos, 'neutral')

def score_orb_close_half(orb_close_h, orb_meaningful):
    if not orb_meaningful or orb_close_h is None: return 'neutral'
    return 'bullish' if orb_close_h == 'upper' else 'bearish'

def score_orb_position(bucket):
    if bucket == 'bottom_third': return 'bullish'
    if bucket == 'top_third':    return 'bearish'
    return 'neutral'

def score_ib_extreme_first(extreme):
    if extreme == 'low':  return 'bullish'
    if extreme == 'high': return 'bearish'
    return 'neutral'

def score_ib_close_half(close_h):
    if close_h == 'upper': return 'bullish'
    if close_h == 'lower': return 'bearish'
    return 'neutral'

def compute_trade_type(ib_extreme, ib_close_h):
    """
    EXTENSION: extreme and close half tell the same directional story.
      low_first  + upper_close → market auctioned up and held → extension long
      high_first + lower_close → market auctioned down and held → extension short
    FADE: market tried one direction, got rejected, closed the other way.
      high_first + upper_close → tried down, rejected, closed upper → fade short / go long
      low_first  + lower_close → tried up, rejected, closed lower  → fade long  / go short
    """
    if ib_extreme is None or ib_close_h is None: return 'unclear'
    if ib_extreme == 'low'  and ib_close_h == 'upper': return 'extension_long'
    if ib_extreme == 'high' and ib_close_h == 'lower': return 'extension_short'
    if ib_extreme == 'high' and ib_close_h == 'upper': return 'fade_long'
    if ib_extreme == 'low'  and ib_close_h == 'lower': return 'fade_short'
    return 'unclear'

def aggregate_signals(scores, min_agreement=4):
    bull = scores.count('bullish')
    bear = scores.count('bearish')
    neut = scores.count('neutral')
    agreement_met = max(bull, bear) >= min_agreement
    if bull == 0 and bear == 0:
        return 'no_edge', bull, bear, neut, False
    bias = 'bullish' if bull >= bear else 'bearish'
    return bias, bull, bear, neut, agreement_met


# ============================================================
# MAIN LOOP — build one record per trading day
# ============================================================

print("\n  Building per-day records...")

records = []
skipped = 0

for idx, date in enumerate(dates[1:], 1):

    if idx % 500 == 0:
        print(f"    Day {idx} of {len(dates)-1}...")

    actual = date + pd.Timedelta(hours=7)

    # ── Time boundaries ──────────────────────────────────────────────────────

    asia_start  = actual.replace(hour=18) - pd.Timedelta(days=1)
    asia_end    = actual.replace(hour=2)
    lon_start   = actual.replace(hour=2)
    lon_end     = actual.replace(hour=8)

    orb5_start  = actual.replace(hour=9,  minute=30)
    orb5_end    = actual.replace(hour=9,  minute=35)   # 5-min ORB end
    ib_start    = actual.replace(hour=9,  minute=30)
    q1_end      = actual.replace(hour=9,  minute=45)   # Q1 end = ORB-15 end
    q2_end      = actual.replace(hour=10, minute=0)
    q3_end      = actual.replace(hour=10, minute=15)
    ib_end      = actual.replace(hour=10, minute=30)
    ny_end      = actual.replace(hour=16, minute=0)

    # ── Slice data ───────────────────────────────────────────────────────────

    asia_data    = df.loc[asia_start:asia_end]
    lon_data     = df.loc[lon_start:lon_end]
    orb5_data    = df.loc[orb5_start:orb5_end]   # 9:30–9:35  (5-min ORB)
    orb15_data   = df.loc[ib_start:q1_end]        # 9:30–9:45  (15-min ORB = Q1)
    q1_data      = df.loc[ib_start:q1_end]        # same slice, kept for naming clarity
    q2_data      = df.loc[q1_end:q2_end]
    q3_data      = df.loc[q2_end:q3_end]
    q4_data      = df.loc[q3_end:ib_end]
    ib_data      = df.loc[ib_start:ib_end]
    post_ib_data = df.loc[ib_end:ny_end]

    # ── Basic validity checks ────────────────────────────────────────────────

    if any(d.empty for d in [asia_data, lon_data, orb5_data, orb15_data,
                              q1_data, q2_data, ib_data, post_ib_data]):
        skipped += 1; continue
    if len(orb5_data) < 5:
        skipped += 1; continue

    # ── Previous day (for range normalization) ───────────────────────────────

    prev_actual = (date - pd.Timedelta(days=1)) + pd.Timedelta(hours=7)
    prev_data   = df.loc[
        prev_actual.replace(hour=18) - pd.Timedelta(days=1)
        : prev_actual.replace(hour=17)
    ]
    prev_range = (prev_data['High'].max() - prev_data['Low'].min()) if not prev_data.empty else None

    # ── ORB-5 metrics ────────────────────────────────────────────────────────

    orb5_open  = orb5_data['Open'].iloc[0]
    orb5_close = orb5_data['Close'].iloc[-1]
    orb5_high  = orb5_data['High'].max()
    orb5_low   = orb5_data['Low'].min()
    orb5_range = orb5_high - orb5_low

    if pd.isna([orb5_open, orb5_close, orb5_high, orb5_low]).any():
        skipped += 1; continue

    orb5_range_meaningful = (
        prev_range is not None and
        prev_range > 0 and
        orb5_range >= prev_range * MIN_ORB_RANGE_FRACTION
    )

    orb5_close_half  = close_half(orb5_close, orb5_high, orb5_low) if orb5_range_meaningful else None
    orb5_direction   = session_direction(orb5_open, orb5_close)
    orb5_mid         = (orb5_high + orb5_low) / 2

    # ── ORB-15 metrics ───────────────────────────────────────────────────────

    orb15_open  = orb15_data['Open'].iloc[0]
    orb15_close = orb15_data['Close'].iloc[-1]
    orb15_high  = orb15_data['High'].max()
    orb15_low   = orb15_data['Low'].min()
    orb15_range = orb15_high - orb15_low

    if pd.isna([orb15_open, orb15_close, orb15_high, orb15_low]).any():
        skipped += 1; continue

    orb15_range_meaningful = (
        prev_range is not None and
        prev_range > 0 and
        orb15_range >= prev_range * MIN_ORB_RANGE_FRACTION
    )

    orb15_close_half = close_half(orb15_close, orb15_high, orb15_low) if orb15_range_meaningful else None
    orb15_direction  = session_direction(orb15_open, orb15_close)
    orb15_mid        = (orb15_high + orb15_low) / 2

    # ── IB metrics ───────────────────────────────────────────────────────────

    ib_open  = ib_data['Open'].iloc[0]
    ib_close = ib_data['Close'].iloc[-1]
    ib_high  = ib_data['High'].max()
    ib_low   = ib_data['Low'].min()
    ib_range = ib_high - ib_low

    if pd.isna([ib_open, ib_close, ib_high, ib_low]).any() or ib_range == 0:
        skipped += 1; continue

    ib_close_half_val = close_half(ib_close, ib_high, ib_low)

    # ── ORB-5 position within IB ─────────────────────────────────────────────

    orb5_pos_in_ib  = position_in_range(orb5_mid, ib_high, ib_low)
    orb5_pos_bucket = position_bucket(orb5_pos_in_ib)

    # ── ORB-15 position within IB ────────────────────────────────────────────

    orb15_pos_in_ib  = position_in_range(orb15_mid, ib_high, ib_low)
    orb15_pos_bucket = position_bucket(orb15_pos_in_ib)

    # ── IB extreme formed first ──────────────────────────────────────────────

    ib_extreme_first_val = None
    for _, bar in ib_data.iterrows():
        hit_high = bar['High'] >= ib_high
        hit_low  = bar['Low']  <= ib_low
        if hit_high and hit_low:
            ib_extreme_first_val = 'simultaneous'; break
        elif hit_high:
            ib_extreme_first_val = 'high'; break
        elif hit_low:
            ib_extreme_first_val = 'low'; break

    # ── Which quarter sets IB high/low ───────────────────────────────────────

    def which_quarter(timestamp):
        if timestamp < q1_end:   return 1
        if timestamp < q2_end:   return 2
        if timestamp < q3_end:   return 3
        return 4

    ib_high_time = ib_data[ib_data['High'] == ib_high].index[0]
    ib_low_time  = ib_data[ib_data['Low']  == ib_low].index[0]
    ib_high_q    = which_quarter(ib_high_time)
    ib_low_q     = which_quarter(ib_low_time)

    ib_high_half = 1 if ib_high_time < q2_end else 2
    ib_low_half  = 1 if ib_low_time  < q2_end else 2

    if ib_high_half != ib_low_half:
        ib_curve_type = 'high_first' if ib_high_half == 1 else 'low_first'
    elif ib_high_half == 1:
        ib_curve_type = 'both_first_half'
    else:
        ib_curve_type = 'both_second_half'

    # ── Q1 / Q2 / second-half ranges ────────────────────────────────────────

    q1_high = q1_data['High'].max()
    q1_low  = q1_data['Low'].min()
    q2_high = q2_data['High'].max()
    q2_low  = q2_data['Low'].min()

    # Second half of IB = Q3 + Q4
    second_half_data = df.loc[q2_end:ib_end]
    if second_half_data.empty:
        skipped += 1; continue

    sh_high = second_half_data['High'].max()
    sh_low  = second_half_data['Low'].min()

    # Third quarter of IB = Q3 (used as "second half" for 15-min ORB curve)
    # For ORB-15, the range covers Q1 (9:30–9:45).
    # The "next quarter" that can break it is Q2 (9:45–10:00).
    # The "remaining IB" that may curve or continue is Q3+Q4 (same as ORB-5 analysis).
    # So the ORB-15 curve analysis uses the same q2/sh windows — just a wider reference range.

    # ── ORB-5 curve analysis ─────────────────────────────────────────────────

    q2_breaks_orb5_high = q2_high > orb5_high
    q2_breaks_orb5_low  = q2_low  < orb5_low

    orb5_curve_result = None
    if q2_breaks_orb5_high and not q2_breaks_orb5_low:
        if sh_low < orb5_low:
            orb5_curve_result = 'broke_high_curved_low'
        elif sh_high > q2_high:
            orb5_curve_result = 'broke_high_continued_high'
        else:
            orb5_curve_result = 'broke_high_neither'
    elif q2_breaks_orb5_low and not q2_breaks_orb5_high:
        if sh_high > orb5_high:
            orb5_curve_result = 'broke_low_curved_high'
        elif sh_low < q2_low:
            orb5_curve_result = 'broke_low_continued_low'
        else:
            orb5_curve_result = 'broke_low_neither'
    elif q2_breaks_orb5_high and q2_breaks_orb5_low:
        orb5_curve_result = 'q2_breaks_both'
    else:
        orb5_curve_result = 'q2_contained'

    # ── ORB-15 curve analysis ─────────────────────────────────────────────────
    #
    # ORB-15 = Q1 (9:30–9:45).  The next candle block that can break it is Q2
    # (9:45–10:00).  The second half of the IB (Q3+Q4) tells us curve vs continue.
    # This is structurally identical to the ORB-5 analysis, just with a wider
    # reference range (orb15_high / orb15_low instead of orb5_high / orb5_low).
    # Note: orb15_high == q1_high and orb15_low == q1_low by definition.

    q2_breaks_orb15_high = q2_high > orb15_high
    q2_breaks_orb15_low  = q2_low  < orb15_low

    orb15_curve_result = None
    if q2_breaks_orb15_high and not q2_breaks_orb15_low:
        if sh_low < orb15_low:
            orb15_curve_result = 'broke_high_curved_low'
        elif sh_high > q2_high:
            orb15_curve_result = 'broke_high_continued_high'
        else:
            orb15_curve_result = 'broke_high_neither'
    elif q2_breaks_orb15_low and not q2_breaks_orb15_high:
        if sh_high > orb15_high:
            orb15_curve_result = 'broke_low_curved_high'
        elif sh_low < q2_low:
            orb15_curve_result = 'broke_low_continued_low'
        else:
            orb15_curve_result = 'broke_low_neither'
    elif q2_breaks_orb15_high and q2_breaks_orb15_low:
        orb15_curve_result = 'q2_breaks_both'
    else:
        orb15_curve_result = 'q2_contained'

    # ── Q1 curve analysis (unchanged, kept for Section 5) ────────────────────

    q2_breaks_q1_high = q2_high > q1_high   # identical to orb15 curve above
    q2_breaks_q1_low  = q2_low  < q1_low

    q1_curve_result = None
    if q2_breaks_q1_high and not q2_breaks_q1_low:
        if sh_low < q1_low:
            q1_curve_result = 'broke_high_curved_low'
        elif sh_high > q2_high:
            q1_curve_result = 'broke_high_continued_high'
        else:
            q1_curve_result = 'broke_high_neither'
    elif q2_breaks_q1_low and not q2_breaks_q1_high:
        if sh_high > q1_high:
            q1_curve_result = 'broke_low_curved_high'
        elif sh_low < q2_low:
            q1_curve_result = 'broke_low_continued_low'
        else:
            q1_curve_result = 'broke_low_neither'
    elif q2_breaks_q1_high and q2_breaks_q1_low:
        q1_curve_result = 'q2_breaks_both'
    else:
        q1_curve_result = 'q2_contained'

    # ── Post-IB direction ────────────────────────────────────────────────────

    post_ib_close = post_ib_data['Close'].iloc[-1]
    post_ib_dir   = session_direction(ib_close, post_ib_close)

    # ── First IB extreme broken post-IB ─────────────────────────────────────
    # After the IB closes (10:30), which boundary gets taken out first?
    # 'high' = IB high broken before IB low; 'low' = IB low broken first.

    first_break_dir = None
    for _, bar in post_ib_data.iterrows():
        broke_high = bar['High'] > ib_high
        broke_low  = bar['Low']  < ib_low
        if broke_high and broke_low:
            first_break_dir = 'simultaneous'; break
        elif broke_high:
            first_break_dir = 'high'; break
        elif broke_low:
            first_break_dir = 'low'; break

    # ── Chain agreement flags ─────────────────────────────────────────────────

    # ORB-5 close half vs IB close half
    orb5_ib_close_agree = (
        orb5_close_half is not None and
        ib_close_half_val is not None and
        orb5_close_half == ib_close_half_val
    )

    # ORB-15 close half vs IB close half
    orb15_ib_close_agree = (
        orb15_close_half is not None and
        ib_close_half_val is not None and
        orb15_close_half == ib_close_half_val
    )

    # IB extreme first vs IB close half
    extreme_close_agree = None
    if ib_extreme_first_val in ('high', 'low') and ib_close_half_val is not None:
        expected_close = 'lower' if ib_extreme_first_val == 'high' else 'upper'
        extreme_close_agree = (ib_close_half_val == expected_close)

    # Full 3-way confluence — ORB-5 version
    full_confluence_5 = None
    if (orb5_close_half is not None and
        orb5_pos_bucket is not None and
        ib_close_half_val is not None):
        bull5 = (orb5_close_half == 'upper' and orb5_pos_bucket == 'bottom_third' and ib_close_half_val == 'upper')
        bear5 = (orb5_close_half == 'lower' and orb5_pos_bucket == 'top_third'    and ib_close_half_val == 'lower')
        full_confluence_5 = 'bullish' if bull5 else ('bearish' if bear5 else 'mixed')

    # Full 3-way confluence — ORB-15 version
    full_confluence_15 = None
    if (orb15_close_half is not None and
        orb15_pos_bucket is not None and
        ib_close_half_val is not None):
        bull15 = (orb15_close_half == 'upper' and orb15_pos_bucket == 'bottom_third' and ib_close_half_val == 'upper')
        bear15 = (orb15_close_half == 'lower' and orb15_pos_bucket == 'top_third'    and ib_close_half_val == 'lower')
        full_confluence_15 = 'bullish' if bull15 else ('bearish' if bear15 else 'mixed')

    # ── Asia metrics ─────────────────────────────────────────────────────────

    asia_open  = asia_data['Open'].iloc[0]
    asia_close = asia_data['Close'].iloc[-1]
    asia_high  = asia_data['High'].max()
    asia_low   = asia_data['Low'].min()
    asia_dir   = session_direction(asia_open, asia_close)

    # ── London metrics ────────────────────────────────────────────────────────

    lon_open  = lon_data['Open'].iloc[0]
    lon_close = lon_data['Close'].iloc[-1]
    lon_high  = lon_data['High'].max()
    lon_low   = lon_data['Low'].min()
    lon_dir   = session_direction(lon_open, lon_close)

    # ── Session pattern ───────────────────────────────────────────────────────

    if asia_dir and lon_dir:
        session_pattern = f"{asia_dir[:4]}_{lon_dir[:4]}"
    else:
        session_pattern = None

    # ── Asia / London positional relationship ─────────────────────────────────

    asia_lon_pos = None
    if not any(pd.isna(v) for v in [asia_high, asia_low, lon_high, lon_low]):
        if   lon_low >= asia_low  and lon_high <= asia_high:
            asia_lon_pos = 'asia_engulfs_london'
        elif asia_low >= lon_low  and asia_high <= lon_high:
            asia_lon_pos = 'london_engulfs_asia'
        elif lon_high > asia_high and lon_low >= asia_low:
            asia_lon_pos = 'london_partial_up'
        elif lon_low  < asia_low  and lon_high <= asia_high:
            asia_lon_pos = 'london_partial_down'
        elif lon_high > asia_high and lon_low < asia_low:
            asia_lon_pos = 'london_breaks_both'
        else:
            asia_lon_pos = 'overlapping'

    # ── Signal scoring ────────────────────────────────────────────────────────

    sig1 = score_session_pattern(session_pattern)
    sig2 = score_asia_london_position(asia_lon_pos)
    sig3 = score_orb_close_half(orb5_close_half, orb5_range_meaningful)
    sig4 = score_orb_position(orb5_pos_bucket)
    sig5 = score_ib_extreme_first(ib_extreme_first_val)
    sig6 = score_ib_close_half(ib_close_half_val)

    signals = [sig1, sig2, sig3, sig4, sig5, sig6]
    bias, bull_count, bear_count, neut_count, agreement_met = aggregate_signals(
        signals, MIN_SIGNAL_AGREEMENT)
    score_label = bias if agreement_met else 'no_edge'

    # ── Trade type ────────────────────────────────────────────────────────────

    trade_type = compute_trade_type(ib_extreme_first_val, ib_close_half_val)

    records.append({
        # Identifiers
        'date'                    : actual.date(),
        'prev_range'              : prev_range,

        # ── Session context ──
        'asia_direction'          : asia_dir,
        'london_direction'        : lon_dir,
        'session_pattern'         : session_pattern,
        'asia_london_pos'         : asia_lon_pos,

        # ── Signal scores ──
        'sig1_session_pattern'    : sig1,
        'sig2_asia_lon_pos'       : sig2,
        'sig3_orb_close_half'     : sig3,
        'sig4_orb_position'       : sig4,
        'sig5_ib_extreme_first'   : sig5,
        'sig6_ib_close_half'      : sig6,
        'bull_signal_count'       : bull_count,
        'bear_signal_count'       : bear_count,
        'neut_signal_count'       : neut_count,
        'score_label'             : score_label,
        'agreement_met'           : agreement_met,
        'trade_type'              : trade_type,

        # ── ORB-5 ──
        'orb5_range'              : orb5_range,
        'orb5_range_meaningful'   : orb5_range_meaningful,
        'orb5_direction'          : orb5_direction,
        'orb5_close_half'         : orb5_close_half,
        'orb5_pos_in_ib'          : orb5_pos_in_ib,
        'orb5_pos_bucket'         : orb5_pos_bucket,
        'orb5_curve_result'       : orb5_curve_result,
        'orb5_ib_close_agree'     : orb5_ib_close_agree,
        'full_confluence_5'       : full_confluence_5,

        # ── ORB-15 ──
        'orb15_range'             : orb15_range,
        'orb15_range_meaningful'  : orb15_range_meaningful,
        'orb15_direction'         : orb15_direction,
        'orb15_close_half'        : orb15_close_half,
        'orb15_pos_in_ib'         : orb15_pos_in_ib,
        'orb15_pos_bucket'        : orb15_pos_bucket,
        'orb15_curve_result'      : orb15_curve_result,
        'orb15_ib_close_agree'    : orb15_ib_close_agree,
        'full_confluence_15'      : full_confluence_15,

        # ── IB structure ──
        'ib_close_half'           : ib_close_half_val,
        'ib_extreme_first'        : ib_extreme_first_val,
        'ib_curve_type'           : ib_curve_type,
        'ib_high_quarter'         : ib_high_q,
        'ib_low_quarter'          : ib_low_q,
        'ib_high_half'            : ib_high_half,
        'ib_low_half'             : ib_low_half,

        # ── Q1 curve (retained for Section 5) ──
        'q1_curve_result'         : q1_curve_result,

        # ── Chain flags ──
        'extreme_close_agree'     : extreme_close_agree,

        # ── Outcome ──
        'post_ib_direction'       : post_ib_dir,
        'first_break_dir'         : first_break_dir,
    })

rdf = pd.DataFrame(records)
total_days    = len(rdf)
valid_post_ib = rdf['post_ib_direction'].notna().sum()

print(f"\n  Records built : {total_days:,}  |  Skipped: {skipped}")
print(f"  Valid post-IB : {valid_post_ib:,}")

# Save full dataset to CSV for external querying
rdf.to_csv(DATASET_CSV_PATH, index=False)
print(f"  Dataset saved : {DATASET_CSV_PATH}")

# Convenience filtered frames
meaningful5  = rdf[rdf['orb5_range_meaningful']  == True]
meaningful15 = rdf[rdf['orb15_range_meaningful'] == True]


# ============================================================
# CURVE LABEL MAPS (shared across ORB-5, ORB-15, Q1)
# ============================================================

curve_labels = {
    'broke_high_curved_low'     : '{ref} HIGH → 2nd half curves LOW   [CURVE]',
    'broke_high_continued_high' : '{ref} HIGH → 2nd half extends HIGH  [CONTINUATION]',
    'broke_high_neither'        : '{ref} HIGH → 2nd half stays in range',
    'broke_low_curved_high'     : '{ref} LOW  → 2nd half curves HIGH   [CURVE]',
    'broke_low_continued_low'   : '{ref} LOW  → 2nd half extends LOW   [CONTINUATION]',
    'broke_low_neither'         : '{ref} LOW  → 2nd half stays in range',
    'q2_breaks_both'            : 'Q2 breaks BOTH {ref} extremes',
    'q2_contained'              : 'Q2 contained within {ref} range',
}

def make_curve_label(result, ref):
    return curve_labels[result].replace('{ref}', f'Q2 breaks {ref}')


# ============================================================
# SECTION 1A — ORB-5 INTERNAL STRUCTURE
# ============================================================

print_header("SECTION 1A — ORB-5 INTERNAL STRUCTURE (9:30–9:35)")
print(f"  Where does the 5-min ORB close within its own range?")
print(f"  (Only meaningful ORBs shown — range ≥ {MIN_ORB_RANGE_FRACTION*100:.0f}% of prev day range)\n")

n_meaningful5 = len(meaningful5)
print(f"  Total days              : {total_days:,}")
print(f"  ORB-5 range meaningful  : {n_meaningful5:,}  ({safe_pct(n_meaningful5, total_days):.1f}%)")
print(f"  ORB-5 range too tight   : {total_days - n_meaningful5:,}  ({safe_pct(total_days - n_meaningful5, total_days):.1f}%)")

print_subheader("ORB-5 Close Half Distribution")
for val in ['upper', 'lower']:
    n = (meaningful5['orb5_close_half'] == val).sum()
    print_row(f"ORB-5 closes {val} half", n, n_meaningful5)

print_subheader("ORB-5 Direction vs Close Half")
for orb_dir in ['bullish', 'bearish']:
    subset = meaningful5[meaningful5['orb5_direction'] == orb_dir]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE: continue
    upper_n = (subset['orb5_close_half'] == 'upper').sum()
    lower_n = (subset['orb5_close_half'] == 'lower').sum()
    print(f"\n  ORB-5 direction = {orb_dir} (n={n})")
    print_row("  Closes upper half", upper_n, n)
    print_row("  Closes lower half", lower_n, n)

print_subheader("ORB-5 Close Half → Post-IB Direction")
print(f"  (Does where the ORB-5 closes predict the RTH session?)\n")
for val in ['upper', 'lower']:
    subset = meaningful5[meaningful5['orb5_close_half'] == val]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(f"ORB-5 closes {val} → Post-IB", bull, bear)


# ============================================================
# SECTION 1B — ORB-15 INTERNAL STRUCTURE
# ============================================================

print_header("SECTION 1B — ORB-15 INTERNAL STRUCTURE (9:30–9:45)")
print(f"  Where does the 15-min ORB (= Q1) close within its own range?")
print(f"  (Only meaningful ORBs shown — range ≥ {MIN_ORB_RANGE_FRACTION*100:.0f}% of prev day range)\n")

n_meaningful15 = len(meaningful15)
print(f"  Total days              : {total_days:,}")
print(f"  ORB-15 range meaningful : {n_meaningful15:,}  ({safe_pct(n_meaningful15, total_days):.1f}%)")
print(f"  ORB-15 range too tight  : {total_days - n_meaningful15:,}  ({safe_pct(total_days - n_meaningful15, total_days):.1f}%)")

print_subheader("ORB-15 Close Half Distribution")
for val in ['upper', 'lower']:
    n = (meaningful15['orb15_close_half'] == val).sum()
    print_row(f"ORB-15 closes {val} half", n, n_meaningful15)

print_subheader("ORB-15 Direction vs Close Half")
for orb_dir in ['bullish', 'bearish']:
    subset = meaningful15[meaningful15['orb15_direction'] == orb_dir]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE: continue
    upper_n = (subset['orb15_close_half'] == 'upper').sum()
    lower_n = (subset['orb15_close_half'] == 'lower').sum()
    print(f"\n  ORB-15 direction = {orb_dir} (n={n})")
    print_row("  Closes upper half", upper_n, n)
    print_row("  Closes lower half", lower_n, n)

print_subheader("ORB-15 Close Half → Post-IB Direction")
print(f"  (Does where the ORB-15 closes predict the RTH session?)\n")
for val in ['upper', 'lower']:
    subset = meaningful15[meaningful15['orb15_close_half'] == val]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(f"ORB-15 closes {val} → Post-IB", bull, bear)

print_subheader("ORB-5 vs ORB-15 Close Half Agreement")
print(f"  (Do the two ORBs agree on direction within the IB?)\n")
agree_n    = (meaningful5['orb5_close_half'] == meaningful5['orb15_close_half']).sum()
disagree_n = (meaningful5['orb5_close_half'] != meaningful5['orb15_close_half']).sum()
total_cmp  = agree_n + disagree_n
print_row("ORB-5 and ORB-15 AGREE",    int(agree_n),    total_cmp)
print_row("ORB-5 and ORB-15 DISAGREE", int(disagree_n), total_cmp)
print()

agree_sub = meaningful5[meaningful5['orb5_close_half'] == meaningful5['orb15_close_half']]
bull = (agree_sub['post_ib_direction'] == 'bullish').sum()
bear = (agree_sub['post_ib_direction'] == 'bearish').sum()
print_outcome_row("  When they AGREE   → Post-IB", bull, bear)

disagree_sub = meaningful5[meaningful5['orb5_close_half'] != meaningful5['orb15_close_half']]
bull = (disagree_sub['post_ib_direction'] == 'bullish').sum()
bear = (disagree_sub['post_ib_direction'] == 'bearish').sum()
print_outcome_row("  When they DISAGREE → Post-IB", bull, bear)


# ============================================================
# SECTION 2A — ORB-5 POSITION WITHIN THE IB
# ============================================================

print_header("SECTION 2A — ORB-5 POSITION WITHIN THE IB")
print(f"  Where does the ORB-5 midpoint sit within the full IB range?")
print(f"  0 = ORB at the very bottom of IB  |  1 = ORB at the very top of IB\n")

print_subheader("ORB-5 Position Bucket Distribution")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    n = (rdf['orb5_pos_bucket'] == bucket).sum()
    print_row(bucket, n, total_days)

print_subheader("ORB-5 Close Half → ORB-5 Position in IB")
for close_h in ['upper', 'lower']:
    subset = meaningful5[meaningful5['orb5_close_half'] == close_h]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE: continue
    print(f"  ORB-5 close half = {close_h} (n={n})")
    for bucket in ['bottom_third', 'middle_third', 'top_third']:
        bn = (subset['orb5_pos_bucket'] == bucket).sum()
        print_row(f"  → ORB-5 position {bucket}", bn, n)
    print()

print_subheader("IB Extreme First → ORB-5 Position in IB")
print(f"  (Auction theory: high forms first → ORB should be at top of IB)\n")
for extreme in ['high', 'low']:
    subset = rdf[rdf['ib_extreme_first'] == extreme]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE: continue
    print(f"  ib_extreme_first = {extreme} (n={n})")
    for bucket in ['bottom_third', 'middle_third', 'top_third']:
        bn = (subset['orb5_pos_bucket'] == bucket).sum()
        print_row(f"  → ORB-5 position {bucket}", bn, n)
    print()

print_subheader("ORB-5 Position Bucket → Post-IB Direction")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    subset = rdf[rdf['orb5_pos_bucket'] == bucket]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(f"ORB-5 in {bucket}", bull, bear)


# ============================================================
# SECTION 2B — ORB-15 POSITION WITHIN THE IB
# ============================================================

print_header("SECTION 2B — ORB-15 POSITION WITHIN THE IB")
print(f"  Where does the ORB-15 midpoint sit within the full IB range?")
print(f"  0 = ORB-15 at the very bottom of IB  |  1 = ORB-15 at the very top of IB\n")

print_subheader("ORB-15 Position Bucket Distribution")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    n = (rdf['orb15_pos_bucket'] == bucket).sum()
    print_row(bucket, n, total_days)

print_subheader("ORB-15 Close Half → ORB-15 Position in IB")
for close_h in ['upper', 'lower']:
    subset = meaningful15[meaningful15['orb15_close_half'] == close_h]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE: continue
    print(f"  ORB-15 close half = {close_h} (n={n})")
    for bucket in ['bottom_third', 'middle_third', 'top_third']:
        bn = (subset['orb15_pos_bucket'] == bucket).sum()
        print_row(f"  → ORB-15 position {bucket}", bn, n)
    print()

print_subheader("IB Extreme First → ORB-15 Position in IB")
print(f"  (Auction theory: high forms first → ORB-15 should be at top of IB)\n")
for extreme in ['high', 'low']:
    subset = rdf[rdf['ib_extreme_first'] == extreme]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE: continue
    print(f"  ib_extreme_first = {extreme} (n={n})")
    for bucket in ['bottom_third', 'middle_third', 'top_third']:
        bn = (subset['orb15_pos_bucket'] == bucket).sum()
        print_row(f"  → ORB-15 position {bucket}", bn, n)
    print()

print_subheader("ORB-15 Position Bucket → Post-IB Direction")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    subset = rdf[rdf['orb15_pos_bucket'] == bucket]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(f"ORB-15 in {bucket}", bull, bear)

print_subheader("ORB-5 vs ORB-15 Position Agreement")
print(f"  (Do both ORBs end up in the same third of the IB?)\n")
pos_agree = (rdf['orb5_pos_bucket'] == rdf['orb15_pos_bucket']).sum()
pos_total = rdf['orb5_pos_bucket'].notna().sum()
print_row("Both ORBs in SAME IB third",      int(pos_agree),             pos_total)
print_row("ORBs in DIFFERENT IB thirds", int(pos_total - pos_agree), pos_total)


# ============================================================
# SECTION 2C — ORB AS PREDICTOR OF IB EXTREME FIRST
# ============================================================
#
# Core question: can we look at the ORB in real time (before the IB
# is complete) and predict whether the IB HIGH or IB LOW will form
# first?
#
# Three predictive angles:
#   (a) ORB close half        → ib_extreme_first
#   (b) ORB position bucket   → ib_extreme_first   [uses final IB — informational]
#   (c) ORB direction         → ib_extreme_first
#   (d) ORB-5 + ORB-15 combined close half → ib_extreme_first
#   (e) ORB close half + direction combined → ib_extreme_first
#   (f) Accuracy summary — how well does each signal predict the extreme?
#
# NOTE on timing:
#   • ORB-5 close half and direction are known at 9:35 — fully real-time
#     before the rest of the IB unfolds.
#   • ORB-15 close half and direction are known at 9:45 — still inside
#     the IB, with Q2–Q4 yet to print.
#   • ORB position in IB uses the *final* IB range so it is retrospective;
#     included here because Section 2A/2B already showed the relationship
#     in the reverse direction, and it completes the picture.
#
# ============================================================

print_header("SECTION 2C — ORB AS PREDICTOR OF IB EXTREME FIRST")
print(f"  Can the ORB tell us — in real time — whether the IB high or low")
print(f"  will form first?")
print(f"  Only 'high' and 'low' ib_extreme_first values used (drops 'simultaneous').\n")

# Filter to rows with a clear extreme_first value
ef = rdf[rdf['ib_extreme_first'].isin(['high', 'low'])].copy()
n_ef = len(ef)
ef5  = ef[ef['orb5_range_meaningful']  == True]
ef15 = ef[ef['orb15_range_meaningful'] == True]

# Baseline distribution of ib_extreme_first
n_high_first = (ef['ib_extreme_first'] == 'high').sum()
n_low_first  = (ef['ib_extreme_first'] == 'low').sum()
print(f"  Total days with clear ib_extreme_first : {n_ef:,}")
print(f"  IB high forms first                    : {n_high_first:,}  ({safe_pct(n_high_first, n_ef):.1f}%)")
print(f"  IB low  forms first                    : {n_low_first:,}  ({safe_pct(n_low_first,  n_ef):.1f}%)")
print(f"  (Baseline: ~50/50 expected if no signal)\n")


def print_extreme_prediction(label, subset, indent=4):
    """Print how well a subset predicts ib_extreme_first."""
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE:
        print(f"{' '*indent}{label:<58} n={n:4d}  (below min sample)")
        return
    n_h = (subset['ib_extreme_first'] == 'high').sum()
    n_l = (subset['ib_extreme_first'] == 'low').sum()
    pct_h = safe_pct(n_h, n)
    pct_l = safe_pct(n_l, n)
    edge  = abs(pct_h - 50)
    bias  = 'HIGH-first' if pct_h > 50 else 'LOW-first'
    print(f"{' '*indent}{label:<58} n={n:4d}  "
          f"High:{pct_h:5.1f}%  Low:{pct_l:5.1f}%  "
          f"Edge:{edge:4.1f}%  [{bias}]")


# ------------------------------------------------------------------
# (a) ORB-5 Close Half → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(a) ORB-5 Close Half → IB Extreme First  [known at 9:35]")
print(f"  Theory: ORB closes upper → market already auctioned up → low still ahead\n")
for val in ['upper', 'lower']:
    subset = ef5[ef5['orb5_close_half'] == val]
    print_extreme_prediction(f"ORB-5 closes {val} half", subset)

# ------------------------------------------------------------------
# (b) ORB-15 Close Half → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(b) ORB-15 Close Half → IB Extreme First  [known at 9:45]")
print(f"  Theory: ORB-15 closes upper → low forms later; closes lower → high forms later\n")
for val in ['upper', 'lower']:
    subset = ef15[ef15['orb15_close_half'] == val]
    print_extreme_prediction(f"ORB-15 closes {val} half", subset)

# ------------------------------------------------------------------
# (c) ORB-5 Direction → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(c) ORB-5 Direction → IB Extreme First  [known at 9:35]")
print(f"  Theory: bullish ORB → upward auction done early → low forms later\n")
for val in ['bullish', 'bearish']:
    subset = ef[ef['orb5_direction'] == val]
    print_extreme_prediction(f"ORB-5 direction = {val}", subset)

# ------------------------------------------------------------------
# (d) ORB-15 Direction → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(d) ORB-15 Direction → IB Extreme First  [known at 9:45]")
print(f"  Theory: bullish Q1 → low still ahead for the IB\n")
for val in ['bullish', 'bearish']:
    subset = ef[ef['orb15_direction'] == val]
    print_extreme_prediction(f"ORB-15 direction = {val}", subset)

# ------------------------------------------------------------------
# (e) ORB-5 Close Half + Direction combined → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(e) ORB-5 Close Half × Direction → IB Extreme First  [known at 9:35]")
print(f"  Strongest signal expected when close half and direction agree\n")
for ch in ['upper', 'lower']:
    for di in ['bullish', 'bearish']:
        subset = ef5[(ef5['orb5_close_half'] == ch) & (ef5['orb5_direction'] == di)]
        print_extreme_prediction(f"ORB-5 closes {ch} + direction {di}", subset)

# ------------------------------------------------------------------
# (f) ORB-15 Close Half + Direction combined → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(f) ORB-15 Close Half × Direction → IB Extreme First  [known at 9:45]")
for ch in ['upper', 'lower']:
    for di in ['bullish', 'bearish']:
        subset = ef15[(ef15['orb15_close_half'] == ch) & (ef15['orb15_direction'] == di)]
        print_extreme_prediction(f"ORB-15 closes {ch} + direction {di}", subset)

# ------------------------------------------------------------------
# (g) ORB-5 + ORB-15 combined close half → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(g) ORB-5 + ORB-15 Both Agree on Close Half → IB Extreme First  [known at 9:45]")
print(f"  Two-timeframe confirmation: do both ORBs closing in the same half\n"
      f"  give a stronger signal about which IB extreme forms first?\n")
# Both upper
both_upper = ef5[(ef5['orb5_close_half'] == 'upper') & (ef5['orb15_close_half'] == 'upper')]
print_extreme_prediction("Both ORB-5 + ORB-15 close UPPER", both_upper)

# Both lower
both_lower = ef5[(ef5['orb5_close_half'] == 'lower') & (ef5['orb15_close_half'] == 'lower')]
print_extreme_prediction("Both ORB-5 + ORB-15 close LOWER", both_lower)

# Disagree: ORB-5 upper, ORB-15 lower
disagree_5u_15l = ef5[(ef5['orb5_close_half'] == 'upper') & (ef5['orb15_close_half'] == 'lower')]
print_extreme_prediction("ORB-5 upper BUT ORB-15 lower (disagree)", disagree_5u_15l)

# Disagree: ORB-5 lower, ORB-15 upper
disagree_5l_15u = ef5[(ef5['orb5_close_half'] == 'lower') & (ef5['orb15_close_half'] == 'upper')]
print_extreme_prediction("ORB-5 lower BUT ORB-15 upper (disagree)", disagree_5l_15u)

# ------------------------------------------------------------------
# (h) ORB position in IB (retrospective) → ib_extreme_first
# ------------------------------------------------------------------
print_subheader("(h) ORB-5 Position in IB → IB Extreme First  [RETROSPECTIVE — uses final IB]")
print(f"  Confirms auction theory; not usable in real time but validates the signal chain.\n")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    subset = ef[ef['orb5_pos_bucket'] == bucket]
    print_extreme_prediction(f"ORB-5 in {bucket}", subset)

print_subheader("(i) ORB-15 Position in IB → IB Extreme First  [RETROSPECTIVE — uses final IB]")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    subset = ef[ef['orb15_pos_bucket'] == bucket]
    print_extreme_prediction(f"ORB-15 in {bucket}", subset)

# ------------------------------------------------------------------
# (j) Accuracy summary — ranked by edge
# ------------------------------------------------------------------
print_subheader("(j) Accuracy Summary — all signals ranked by edge over 50% baseline")
print(f"  How accurately does each real-time ORB signal predict ib_extreme_first?\n")

accuracy_rows = []

def acc_collect(label, subset, timing):
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE:
        return
    n_h   = (subset['ib_extreme_first'] == 'high').sum()
    pct_h = safe_pct(n_h, n)
    edge  = abs(pct_h - 50)
    bias  = 'HIGH-first' if pct_h > 50 else 'LOW-first'
    accuracy_rows.append({
        'label'  : label,
        'timing' : timing,
        'n'      : n,
        'pct_h'  : pct_h,
        'pct_l'  : 100 - pct_h,
        'edge'   : edge,
        'bias'   : bias,
    })

# ORB-5 signals (known at 9:35)
for val in ['upper', 'lower']:
    acc_collect(f"ORB-5 closes {val}",      ef5[ef5['orb5_close_half'] == val],   '9:35')
for val in ['bullish', 'bearish']:
    acc_collect(f"ORB-5 direction {val}",   ef[ef['orb5_direction'] == val],      '9:35')
for ch in ['upper', 'lower']:
    for di in ['bullish', 'bearish']:
        acc_collect(f"ORB-5 {ch}+{di}",
                    ef5[(ef5['orb5_close_half'] == ch) & (ef5['orb5_direction'] == di)], '9:35')

# ORB-15 signals (known at 9:45)
for val in ['upper', 'lower']:
    acc_collect(f"ORB-15 closes {val}",     ef15[ef15['orb15_close_half'] == val], '9:45')
for val in ['bullish', 'bearish']:
    acc_collect(f"ORB-15 direction {val}",  ef[ef['orb15_direction'] == val],      '9:45')
for ch in ['upper', 'lower']:
    for di in ['bullish', 'bearish']:
        acc_collect(f"ORB-15 {ch}+{di}",
                    ef15[(ef15['orb15_close_half'] == ch) & (ef15['orb15_direction'] == di)], '9:45')

# Two-timeframe combos (known at 9:45)
acc_collect("Both ORBs close UPPER",  ef5[(ef5['orb5_close_half']=='upper')&(ef5['orb15_close_half']=='upper')], '9:45')
acc_collect("Both ORBs close LOWER",  ef5[(ef5['orb5_close_half']=='lower')&(ef5['orb15_close_half']=='lower')], '9:45')
acc_collect("ORB-5 upper, ORB-15 lower (split)", disagree_5u_15l, '9:45')
acc_collect("ORB-5 lower, ORB-15 upper (split)", disagree_5l_15u, '9:45')

acc_df = pd.DataFrame(accuracy_rows).sort_values('edge', ascending=False)

print(f"  {'SIGNAL':<42} {'TIMING':>6}  {'n':>6}  "
      f"{'HIGH%':>6}  {'LOW%':>6}  {'EDGE':>6}  {'PREDICTS':>10}")
print(f"  {'-'*42}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*10}")

for _, row in acc_df.iterrows():
    print(f"  {row['label']:<42}  {row['timing']:>6}  {row['n']:>6}  "
          f"{row['pct_h']:>5.1f}%  {row['pct_l']:>5.1f}%  "
          f"{row['edge']:>5.1f}%  {row['bias']:>10}")


# ============================================================
# SECTION 3 — IB QUARTER STRUCTURE  (unchanged)
# ============================================================

print_header("SECTION 3 — IB QUARTER STRUCTURE")
print(f"  15-minute quarter breakdown of the IB (9:30–10:30am)\n")

print_subheader("Which Quarter Sets the IB High?")
for q in [1, 2, 3, 4]:
    n = (rdf['ib_high_quarter'] == q).sum()
    label = {1: 'Q1 (9:30–9:45)', 2: 'Q2 (9:45–10:00)',
             3: 'Q3 (10:00–10:15)', 4: 'Q4 (10:15–10:30)'}[q]
    print_row(label, n, total_days)

print_subheader("Which Quarter Sets the IB Low?")
for q in [1, 2, 3, 4]:
    n = (rdf['ib_low_quarter'] == q).sum()
    label = {1: 'Q1 (9:30–9:45)', 2: 'Q2 (9:45–10:00)',
             3: 'Q3 (10:00–10:15)', 4: 'Q4 (10:15–10:30)'}[q]
    print_row(label, n, total_days)

print_subheader("IB Curve Type — High/Low Half Distribution")
for ct in ['high_first', 'low_first', 'both_first_half', 'both_second_half']:
    n = (rdf['ib_curve_type'] == ct).sum()
    print_row(ct, n, total_days)

print_subheader("IB Curve Type → Post-IB Direction")
for ct in ['high_first', 'low_first', 'both_first_half', 'both_second_half']:
    subset = rdf[rdf['ib_curve_type'] == ct]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(ct, bull, bear)

print_subheader("IB Close Half Distribution")
for val in ['upper', 'lower']:
    n = (rdf['ib_close_half'] == val).sum()
    print_row(f"IB closes {val} half", n, total_days)

print_subheader("IB Close Half → Post-IB Direction")
for val in ['upper', 'lower']:
    subset = rdf[rdf['ib_close_half'] == val]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(f"IB close half = {val}", bull, bear)


# ============================================================
# SECTION 4A — ORB-5 CURVE ANALYSIS
# ============================================================

print_header("SECTION 4A — ORB-5 CURVE ANALYSIS")
print(f"  When Q2 (9:45–10:00) breaks the ORB-5 range (9:30–9:35),")
print(f"  does the second half of the IB CURVE BACK or CONTINUE?\n")

def print_curve_section(col, ref_label, rdf_frame):
    """Generic printer for any curve result column."""
    results_order = [
        'broke_high_curved_low', 'broke_high_continued_high', 'broke_high_neither',
        'broke_low_curved_high', 'broke_low_continued_low',  'broke_low_neither',
        'q2_breaks_both', 'q2_contained',
    ]
    labels = {r: make_curve_label(r, ref_label) for r in results_order}

    print_subheader(f"{ref_label} Curve Result Distribution")
    for result in results_order:
        n = (rdf_frame[col] == result).sum()
        print_row(labels[result], n, len(rdf_frame))

    print_subheader(f"When Q2 Breaks {ref_label} HIGH — Breakdown")
    broke_high = rdf_frame[rdf_frame[col].isin(
        ['broke_high_curved_low', 'broke_high_continued_high', 'broke_high_neither'])]
    n_bh = len(broke_high)
    if n_bh >= MIN_DISPLAY_SAMPLE:
        for result in ['broke_high_curved_low', 'broke_high_continued_high', 'broke_high_neither']:
            n = (rdf_frame[col] == result).sum()
            lbl = result.replace('broke_high_', '').replace('_', ' ').upper()
            print_row(f"  {lbl}", n, n_bh)

    print_subheader(f"When Q2 Breaks {ref_label} LOW — Breakdown")
    broke_low = rdf_frame[rdf_frame[col].isin(
        ['broke_low_curved_high', 'broke_low_continued_low', 'broke_low_neither'])]
    n_bl = len(broke_low)
    if n_bl >= MIN_DISPLAY_SAMPLE:
        for result in ['broke_low_curved_high', 'broke_low_continued_low', 'broke_low_neither']:
            n = (rdf_frame[col] == result).sum()
            lbl = result.replace('broke_low_', '').replace('_', ' ').upper()
            print_row(f"  {lbl}", n, n_bl)

    print_subheader(f"{ref_label} Curve Result → Post-IB Direction")
    for result in results_order:
        subset = rdf_frame[rdf_frame[col] == result]
        bull = (subset['post_ib_direction'] == 'bullish').sum()
        bear = (subset['post_ib_direction'] == 'bearish').sum()
        print_outcome_row(labels[result][:55], bull, bear)

print_curve_section('orb5_curve_result', 'ORB-5', rdf)


# ============================================================
# SECTION 4B — ORB-15 CURVE ANALYSIS
# ============================================================

print_header("SECTION 4B — ORB-15 CURVE ANALYSIS")
print(f"  When Q2 (9:45–10:00) breaks the ORB-15 range (9:30–9:45),")
print(f"  does the second half of the IB CURVE BACK or CONTINUE?")
print(f"  (ORB-15 = Q1, so this is equivalent to the Q1 Curve Analysis in Section 5)")
print(f"  Presented here with ORB-15 framing for direct comparison with ORB-5.\n")

print_curve_section('orb15_curve_result', 'ORB-15', rdf)

print_subheader("ORB-5 vs ORB-15 Curve Result Agreement")
print(f"  (When Q2 breaks both ORBs in the same direction, does that strengthen the signal?)\n")

same_dir_high = rdf[
    (rdf['orb5_curve_result']  == 'broke_high_continued_high') &
    (rdf['orb15_curve_result'] == 'broke_high_continued_high')
]
bull = (same_dir_high['post_ib_direction'] == 'bullish').sum()
bear = (same_dir_high['post_ib_direction'] == 'bearish').sum()
print_outcome_row("Both continue HIGH (ORB-5 & ORB-15)", bull, bear)

same_dir_low = rdf[
    (rdf['orb5_curve_result']  == 'broke_low_continued_low') &
    (rdf['orb15_curve_result'] == 'broke_low_continued_low')
]
bull = (same_dir_low['post_ib_direction'] == 'bullish').sum()
bear = (same_dir_low['post_ib_direction'] == 'bearish').sum()
print_outcome_row("Both continue LOW  (ORB-5 & ORB-15)", bull, bear)

curve_high_5_cont_15 = rdf[
    (rdf['orb5_curve_result']  == 'broke_high_curved_low') &
    (rdf['orb15_curve_result'] == 'broke_high_continued_high')
]
bull = (curve_high_5_cont_15['post_ib_direction'] == 'bullish').sum()
bear = (curve_high_5_cont_15['post_ib_direction'] == 'bearish').sum()
print_outcome_row("ORB-5 curves low BUT ORB-15 continues high", bull, bear)

cont_high_5_curve_15 = rdf[
    (rdf['orb5_curve_result']  == 'broke_high_continued_high') &
    (rdf['orb15_curve_result'] == 'broke_high_curved_low')
]
bull = (cont_high_5_curve_15['post_ib_direction'] == 'bullish').sum()
bear = (cont_high_5_curve_15['post_ib_direction'] == 'bearish').sum()
print_outcome_row("ORB-5 continues high BUT ORB-15 curves low", bull, bear)


# ============================================================
# SECTION 5 — Q1 CURVE ANALYSIS  (unchanged, for reference)
# ============================================================

print_header("SECTION 5 — Q1 CURVE ANALYSIS")
print(f"  When Q2 (9:45–10:00) breaks the Q1 range (9:30–9:45),")
print(f"  does the second half of the IB CURVE BACK or CONTINUE?")
print(f"  (Identical to Section 4B by construction — retained for historical continuity)\n")

q1_curve_labels_map = {r: make_curve_label(r, 'Q1') for r in [
    'broke_high_curved_low', 'broke_high_continued_high', 'broke_high_neither',
    'broke_low_curved_high', 'broke_low_continued_low',  'broke_low_neither',
    'q2_breaks_both', 'q2_contained',
]}

print_subheader("Q1 Curve Result Distribution")
for result, label in q1_curve_labels_map.items():
    n = (rdf['q1_curve_result'] == result).sum()
    print_row(label, n, total_days)

print_subheader("When Q2 Breaks Q1 HIGH — Breakdown")
broke_high_q1 = rdf[rdf['q1_curve_result'].isin(
    ['broke_high_curved_low', 'broke_high_continued_high', 'broke_high_neither'])]
n_bh_q1 = len(broke_high_q1)
if n_bh_q1 >= MIN_DISPLAY_SAMPLE:
    for result in ['broke_high_curved_low', 'broke_high_continued_high', 'broke_high_neither']:
        n = (rdf['q1_curve_result'] == result).sum()
        lbl = result.replace('broke_high_', '').replace('_', ' ').upper()
        print_row(f"  {lbl}", n, n_bh_q1)

print_subheader("When Q2 Breaks Q1 LOW — Breakdown")
broke_low_q1 = rdf[rdf['q1_curve_result'].isin(
    ['broke_low_curved_high', 'broke_low_continued_low', 'broke_low_neither'])]
n_bl_q1 = len(broke_low_q1)
if n_bl_q1 >= MIN_DISPLAY_SAMPLE:
    for result in ['broke_low_curved_high', 'broke_low_continued_low', 'broke_low_neither']:
        n = (rdf['q1_curve_result'] == result).sum()
        lbl = result.replace('broke_low_', '').replace('_', ' ').upper()
        print_row(f"  {lbl}", n, n_bl_q1)

print_subheader("Q1 Curve Result → Post-IB Direction")
for result, label in q1_curve_labels_map.items():
    subset = rdf[rdf['q1_curve_result'] == result]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(label[:55], bull, bear)


# ============================================================
# SECTION 6 — CHAIN VALIDATION
# ============================================================

print_header("SECTION 6 — CHAIN VALIDATION")
print(f"  Testing internal consistency of the auction chain:\n")
print(f"  orb5_close_half / orb15_close_half → orb_position_in_ib → ib_extreme_first → ib_close_half\n")

print_subheader("Link 1a: ORB-5 Close Half → IB Close Half Agreement")
agree5_n    = rdf['orb5_ib_close_agree'].sum()
disagree5_n = (rdf['orb5_ib_close_agree'] == False).sum()
total_l1a   = agree5_n + disagree5_n
print_row("ORB-5 close AGREES with IB close half",    int(agree5_n),    total_l1a)
print_row("ORB-5 close DISAGREES with IB close half", int(disagree5_n), total_l1a)

agree5_sub = rdf[rdf['orb5_ib_close_agree'] == True]
bull = (agree5_sub['post_ib_direction'] == 'bullish').sum()
bear = (agree5_sub['post_ib_direction'] == 'bearish').sum()
print()
print_outcome_row("  ORB-5 / IB agree   → Post-IB", bull, bear)

disagree5_sub = rdf[rdf['orb5_ib_close_agree'] == False]
bull = (disagree5_sub['post_ib_direction'] == 'bullish').sum()
bear = (disagree5_sub['post_ib_direction'] == 'bearish').sum()
print_outcome_row("  ORB-5 / IB disagree → Post-IB", bull, bear)

print_subheader("Link 1b: ORB-15 Close Half → IB Close Half Agreement")
agree15_n    = rdf['orb15_ib_close_agree'].sum()
disagree15_n = (rdf['orb15_ib_close_agree'] == False).sum()
total_l1b    = agree15_n + disagree15_n
print_row("ORB-15 close AGREES with IB close half",    int(agree15_n),    total_l1b)
print_row("ORB-15 close DISAGREES with IB close half", int(disagree15_n), total_l1b)

agree15_sub = rdf[rdf['orb15_ib_close_agree'] == True]
bull = (agree15_sub['post_ib_direction'] == 'bullish').sum()
bear = (agree15_sub['post_ib_direction'] == 'bearish').sum()
print()
print_outcome_row("  ORB-15 / IB agree   → Post-IB", bull, bear)

disagree15_sub = rdf[rdf['orb15_ib_close_agree'] == False]
bull = (disagree15_sub['post_ib_direction'] == 'bullish').sum()
bear = (disagree15_sub['post_ib_direction'] == 'bearish').sum()
print_outcome_row("  ORB-15 / IB disagree → Post-IB", bull, bear)

print_subheader("Link 2: IB Extreme First → IB Close Half Agreement")
print(f"  Theory: high forms first → IB closes lower half (auctioned up, returned)\n")
for extreme in ['high', 'low']:
    subset = rdf[rdf['ib_extreme_first'] == extreme]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE: continue
    expected = 'lower' if extreme == 'high' else 'upper'
    match_n    = (subset['ib_close_half'] == expected).sum()
    mismatch_n = n - match_n
    print(f"  ib_extreme_first = {extreme} (n={n}) — expects ib_close_half = {expected}")
    print_row(f"  ib_close_half = {expected} (matches theory)", match_n, n)
    print_row(f"  ib_close_half = {'upper' if expected == 'lower' else 'lower'} (against theory)", mismatch_n, n)
    print()

print_subheader("Link 3: ib_extreme_first × ib_close_half → Post-IB")
for extreme in ['high', 'low']:
    for close_h in ['upper', 'lower']:
        subset = rdf[(rdf['ib_extreme_first'] == extreme) & (rdf['ib_close_half'] == close_h)]
        bull = (subset['post_ib_direction'] == 'bullish').sum()
        bear = (subset['post_ib_direction'] == 'bearish').sum()
        print_outcome_row(f"extreme_first={extreme} + ib_close={close_h}", bull, bear)

print_subheader("Link 4a: ORB-5 Position Bucket × IB Close Half → Post-IB")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    for close_h in ['upper', 'lower']:
        subset = rdf[(rdf['orb5_pos_bucket'] == bucket) & (rdf['ib_close_half'] == close_h)]
        bull = (subset['post_ib_direction'] == 'bullish').sum()
        bear = (subset['post_ib_direction'] == 'bearish').sum()
        print_outcome_row(f"orb5_pos={bucket} + ib_close={close_h}", bull, bear)

print_subheader("Link 4b: ORB-15 Position Bucket × IB Close Half → Post-IB")
for bucket in ['bottom_third', 'middle_third', 'top_third']:
    for close_h in ['upper', 'lower']:
        subset = rdf[(rdf['orb15_pos_bucket'] == bucket) & (rdf['ib_close_half'] == close_h)]
        bull = (subset['post_ib_direction'] == 'bullish').sum()
        bear = (subset['post_ib_direction'] == 'bearish').sum()
        print_outcome_row(f"orb15_pos={bucket} + ib_close={close_h}", bull, bear)


# ============================================================
# SECTION 7 — POST-IB SESSION OUTCOMES
# ============================================================

print_header("SECTION 7 — POST-IB SESSION OUTCOMES")
print(f"  RTH direction (10:30am → 4:00pm) conditioned on IB structure patterns\n")

baseline_bull  = (rdf['post_ib_direction'] == 'bullish').sum()
baseline_bear  = (rdf['post_ib_direction'] == 'bearish').sum()
baseline_total = baseline_bull + baseline_bear
print(f"  Baseline (all days): Bull {safe_pct(baseline_bull, baseline_total):.1f}%  "
      f"Bear {safe_pct(baseline_bear, baseline_total):.1f}%  (n={baseline_total})\n")

print_subheader("7A. ORB-5 Curve + IB Close Half (2-factor)")
for orb_res in ['broke_high_curved_low', 'broke_high_continued_high',
                 'broke_low_curved_high', 'broke_low_continued_low']:
    for close_h in ['upper', 'lower']:
        subset = rdf[(rdf['orb5_curve_result'] == orb_res) & (rdf['ib_close_half'] == close_h)]
        bull = (subset['post_ib_direction'] == 'bullish').sum()
        bear = (subset['post_ib_direction'] == 'bearish').sum()
        print_outcome_row(f"{orb_res[:28]}(5) + ib_close={close_h}", bull, bear)

print_subheader("7B. ORB-15 Curve + IB Close Half (2-factor)")
for orb_res in ['broke_high_curved_low', 'broke_high_continued_high',
                 'broke_low_curved_high', 'broke_low_continued_low']:
    for close_h in ['upper', 'lower']:
        subset = rdf[(rdf['orb15_curve_result'] == orb_res) & (rdf['ib_close_half'] == close_h)]
        bull = (subset['post_ib_direction'] == 'bullish').sum()
        bear = (subset['post_ib_direction'] == 'bearish').sum()
        print_outcome_row(f"{orb_res[:27]}(15) + ib_close={close_h}", bull, bear)

print_subheader("7C. ORB-5 Close Half + ORB-5 Position + IB Close Half (3-factor)")
for orb_close in ['upper', 'lower']:
    for bucket in ['bottom_third', 'top_third']:
        for ib_close in ['upper', 'lower']:
            subset = meaningful5[
                (meaningful5['orb5_close_half'] == orb_close) &
                (meaningful5['orb5_pos_bucket'] == bucket) &
                (meaningful5['ib_close_half']   == ib_close)
            ]
            bull = (subset['post_ib_direction'] == 'bullish').sum()
            bear = (subset['post_ib_direction'] == 'bearish').sum()
            print_outcome_row(f"orb5={orb_close}+{bucket[:3]}+ib={ib_close}", bull, bear)

print_subheader("7D. ORB-15 Close Half + ORB-15 Position + IB Close Half (3-factor)")
for orb_close in ['upper', 'lower']:
    for bucket in ['bottom_third', 'top_third']:
        for ib_close in ['upper', 'lower']:
            subset = meaningful15[
                (meaningful15['orb15_close_half'] == orb_close) &
                (meaningful15['orb15_pos_bucket'] == bucket) &
                (meaningful15['ib_close_half']    == ib_close)
            ]
            bull = (subset['post_ib_direction'] == 'bullish').sum()
            bear = (subset['post_ib_direction'] == 'bearish').sum()
            print_outcome_row(f"orb15={orb_close}+{bucket[:3]}+ib={ib_close}", bull, bear)

print_subheader("7E. IB Curve Type + IB Close Half + ORB-5 Position (3-factor)")
for ct in ['high_first', 'low_first']:
    for close_h in ['upper', 'lower']:
        for bucket in ['bottom_third', 'top_third']:
            subset = rdf[
                (rdf['ib_curve_type']  == ct) &
                (rdf['ib_close_half']  == close_h) &
                (rdf['orb5_pos_bucket'] == bucket)
            ]
            bull = (subset['post_ib_direction'] == 'bullish').sum()
            bear = (subset['post_ib_direction'] == 'bearish').sum()
            print_outcome_row(f"curve={ct[:4]}+ib={close_h[:3]}+orb5={bucket[:3]}", bull, bear)

print_subheader("7F. IB Curve Type + IB Close Half + ORB-15 Position (3-factor)")
for ct in ['high_first', 'low_first']:
    for close_h in ['upper', 'lower']:
        for bucket in ['bottom_third', 'top_third']:
            subset = rdf[
                (rdf['ib_curve_type']   == ct) &
                (rdf['ib_close_half']   == close_h) &
                (rdf['orb15_pos_bucket'] == bucket)
            ]
            bull = (subset['post_ib_direction'] == 'bullish').sum()
            bear = (subset['post_ib_direction'] == 'bearish').sum()
            print_outcome_row(f"curve={ct[:4]}+ib={close_h[:3]}+orb15={bucket[:3]}", bull, bear)

print_subheader("7G. ORB-5 + ORB-15 Curve Agreement × IB Close Half (confluence)")
for close_h in ['upper', 'lower']:
    # Both continue same direction high
    subset = rdf[
        (rdf['orb5_curve_result']  == 'broke_high_continued_high') &
        (rdf['orb15_curve_result'] == 'broke_high_continued_high') &
        (rdf['ib_close_half']      == close_h)
    ]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(f"both_cont_high + ib_close={close_h}", bull, bear)

    # Both continue same direction low
    subset = rdf[
        (rdf['orb5_curve_result']  == 'broke_low_continued_low') &
        (rdf['orb15_curve_result'] == 'broke_low_continued_low') &
        (rdf['ib_close_half']      == close_h)
    ]
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    print_outcome_row(f"both_cont_low  + ib_close={close_h}", bull, bear)


# ============================================================
# SECTION 8 — FULL CONFLUENCE SUMMARY
# ============================================================

print_header("SECTION 8 — FULL CONFLUENCE SUMMARY")
print(f"  Ranked edge table — all patterns (ORB-5 and ORB-15), sorted by edge over baseline\n")
print(f"  Baseline: Bull {safe_pct(baseline_bull, baseline_total):.1f}%  |  "
      f"Only showing n ≥ {MIN_DISPLAY_SAMPLE}\n")

confluence_results = []

def collect(label, subset):
    bull  = (subset['post_ib_direction'] == 'bullish').sum()
    bear  = (subset['post_ib_direction'] == 'bearish').sum()
    total = bull + bear
    if total < MIN_DISPLAY_SAMPLE:
        return
    bull_pct = safe_pct(bull, total)
    edge = abs(bull_pct - safe_pct(baseline_bull, baseline_total))
    bias = 'BULL' if bull_pct > 50 else 'BEAR'
    confluence_results.append({
        'label'   : label,
        'n'       : total,
        'bull_pct': bull_pct,
        'bear_pct': 100 - bull_pct,
        'edge'    : edge,
        'bias'    : bias,
    })

# ── Single features ──────────────────────────────────────────

for val in ['upper', 'lower']:
    collect(f"orb5_close_half={val}",  meaningful5[meaningful5['orb5_close_half']   == val])
    collect(f"orb15_close_half={val}", meaningful15[meaningful15['orb15_close_half'] == val])
    collect(f"ib_close_half={val}",    rdf[rdf['ib_close_half'] == val])

for bucket in ['bottom_third', 'middle_third', 'top_third']:
    collect(f"orb5_pos={bucket}",  rdf[rdf['orb5_pos_bucket']  == bucket])
    collect(f"orb15_pos={bucket}", rdf[rdf['orb15_pos_bucket'] == bucket])

for extreme in ['high', 'low']:
    collect(f"ib_extreme_first={extreme}", rdf[rdf['ib_extreme_first'] == extreme])

for ct in ['high_first', 'low_first', 'both_first_half', 'both_second_half']:
    collect(f"ib_curve_type={ct}", rdf[rdf['ib_curve_type'] == ct])

for result in [
    'broke_high_curved_low', 'broke_high_continued_high', 'broke_high_neither',
    'broke_low_curved_high', 'broke_low_continued_low',  'broke_low_neither',
    'q2_breaks_both', 'q2_contained',
]:
    collect(f"orb5_curve={result}",  rdf[rdf['orb5_curve_result']  == result])
    collect(f"orb15_curve={result}", rdf[rdf['orb15_curve_result'] == result])
    collect(f"q1_curve={result}",    rdf[rdf['q1_curve_result']    == result])

# ── 2-factor combos ──────────────────────────────────────────

for close_h in ['upper', 'lower']:
    for bucket in ['bottom_third', 'top_third']:
        collect(f"ib_close={close_h}+orb5_pos={bucket}",
                rdf[(rdf['ib_close_half'] == close_h) & (rdf['orb5_pos_bucket']  == bucket)])
        collect(f"ib_close={close_h}+orb15_pos={bucket}",
                rdf[(rdf['ib_close_half'] == close_h) & (rdf['orb15_pos_bucket'] == bucket)])

for extreme in ['high', 'low']:
    for close_h in ['upper', 'lower']:
        collect(f"extreme_first={extreme}+ib_close={close_h}",
                rdf[(rdf['ib_extreme_first'] == extreme) & (rdf['ib_close_half'] == close_h)])

for ct in ['high_first', 'low_first']:
    for close_h in ['upper', 'lower']:
        collect(f"curve_type={ct}+ib_close={close_h}",
                rdf[(rdf['ib_curve_type'] == ct) & (rdf['ib_close_half'] == close_h)])

# ── 3-factor combos ──────────────────────────────────────────

for orb_close in ['upper', 'lower']:
    for bucket in ['bottom_third', 'top_third']:
        for ib_close in ['upper', 'lower']:
            collect(f"orb5={orb_close}+{bucket}+ib={ib_close}",
                    meaningful5[
                        (meaningful5['orb5_close_half'] == orb_close) &
                        (meaningful5['orb5_pos_bucket'] == bucket) &
                        (meaningful5['ib_close_half']   == ib_close)
                    ])
            collect(f"orb15={orb_close}+{bucket}+ib={ib_close}",
                    meaningful15[
                        (meaningful15['orb15_close_half'] == orb_close) &
                        (meaningful15['orb15_pos_bucket'] == bucket) &
                        (meaningful15['ib_close_half']    == ib_close)
                    ])

# Full confluence flags
for fc in ['bullish', 'bearish', 'mixed']:
    collect(f"full_confluence_5={fc}",  rdf[rdf['full_confluence_5']  == fc])
    collect(f"full_confluence_15={fc}", rdf[rdf['full_confluence_15'] == fc])

# ── Sort and print ────────────────────────────────────────────

results_df = pd.DataFrame(confluence_results).sort_values('edge', ascending=False)

print(f"  {'PATTERN':<62} {'n':>6}  {'BULL%':>6}  {'BEAR%':>6}  {'EDGE':>6}  {'BIAS':>5}")
print(f"  {'-'*62}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*5}")

for _, row in results_df.iterrows():
    print(f"  {row['label']:<62} {row['n']:>6}  "
          f"{row['bull_pct']:>5.1f}%  {row['bear_pct']:>5.1f}%  "
          f"{row['edge']:>5.1f}%  {row['bias']:>5}")



# ============================================================
# SECTION 9 — REVERSE CONDITIONAL ANALYSIS
# ============================================================
#
# All previous sections asked: given ORB structure → what happens?
# This section flips the question:
#
#   Given that the IB HIGH (or LOW) was broken first post-IB,
#   what was the ORB position distribution?
#
# Why this matters:
#   The forward question tells you predictive accuracy.
#   The reverse question tells you diagnostic reliability —
#   i.e. if you see a high break, how confident can you be
#   the ORB was set up in a particular way?
#
#   Together they form the full probabilistic picture.
#
# NOTE: first_break_dir is the first IB boundary broken AFTER
#       the IB closes (10:30am). It is distinct from
#       ib_extreme_first (which extreme formed DURING the IB).
#
# ============================================================

print_header("SECTION 9 — REVERSE CONDITIONAL ANALYSIS")
print(f"""
  Question: Given that the IB high or low was broken first after 10:30,
            what was the ORB position distribution?

  Forward (Sections 2A/2B): orb_pos=X → how often does high break first?
  Reverse (this section)  : first_break=high → how often was orb_pos=X?

  This tests diagnostic reliability: when you observe a break,
  how strongly does it retrodict the ORB setup?
""")

# Filter to clean first-break rows
fb  = rdf[rdf['first_break_dir'].isin(['high', 'low'])].copy()
n_fb   = len(fb)
n_hi   = (fb['first_break_dir'] == 'high').sum()
n_lo   = (fb['first_break_dir'] == 'low').sum()
base_hi_fb = safe_pct(n_hi, n_fb)
base_lo_fb = safe_pct(n_lo, n_fb)

buckets = ['bottom_third', 'middle_third', 'top_third']
base_buck5  = {b: safe_pct((fb['orb5_pos_bucket']  == b).sum(), n_fb) for b in buckets}
base_buck15 = {b: safe_pct((fb['orb15_pos_bucket'] == b).sum(), n_fb) for b in buckets}

print(f"  Days with clear first break : {n_fb:,}")
print(f"  First break HIGH            : {n_hi:,}  ({base_hi_fb:.1f}%)")
print(f"  First break LOW             : {n_lo:,}  ({base_lo_fb:.1f}%)")
print(f"\n  ORB-5 position base rates  : "
      f"bot={base_buck5['bottom_third']:.1f}%  "
      f"mid={base_buck5['middle_third']:.1f}%  "
      f"top={base_buck5['top_third']:.1f}%")
print(f"  ORB-15 position base rates : "
      f"bot={base_buck15['bottom_third']:.1f}%  "
      f"mid={base_buck15['middle_third']:.1f}%  "
      f"top={base_buck15['top_third']:.1f}%")


def pct_bar(pct, width=35):
    filled = int(pct / 100 * width)
    return "█" * filled + "░" * (width - filled)


def print_reverse_table(subset, base_buck, orb_label):
    n = len(subset)
    print(f"\n  {'ORB Position':<20}  {'Count':>6}  {'%':>6}  {'vs Base':>8}  Bar")
    print(f"  {'-'*20}  {'-'*6}  {'-'*6}  {'-'*8}  {'-'*35}")
    for bucket in buckets:
        count   = (subset[orb_label] == bucket).sum()
        pct     = safe_pct(count, n)
        vs_base = pct - base_buck[bucket]
        flag    = f"+{vs_base:.1f}%" if vs_base >= 0 else f"{vs_base:.1f}%"
        bar     = pct_bar(pct)
        print(f"  {bucket:<20}  {count:>6}  {pct:>5.1f}%  {flag:>8}  {bar}")


# ------------------------------------------------------------------
# 9A. First break direction → ORB-5 position
# ------------------------------------------------------------------
print_subheader("9A. First Break Direction → ORB-5 Position  [forward: known at 9:35]")

for break_dir in ['high', 'low']:
    subset = fb[fb['first_break_dir'] == break_dir]
    n = len(subset)
    base = base_hi_fb if break_dir == 'high' else base_lo_fb
    print(f"\n  First break = {break_dir.upper()}  (n={n}, base rate {base:.1f}%)")
    print_reverse_table(subset, base_buck5, 'orb5_pos_bucket')


# ------------------------------------------------------------------
# 9B. First break direction → ORB-15 position
# ------------------------------------------------------------------
print_subheader("9B. First Break Direction → ORB-15 Position  [forward: known at 9:45]")

for break_dir in ['high', 'low']:
    subset = fb[fb['first_break_dir'] == break_dir]
    n = len(subset)
    base = base_hi_fb if break_dir == 'high' else base_lo_fb
    print(f"\n  First break = {break_dir.upper()}  (n={n}, base rate {base:.1f}%)")
    print_reverse_table(subset, base_buck15, 'orb15_pos_bucket')


# ------------------------------------------------------------------
# 9C. Full cross-tab with lift scores
# ------------------------------------------------------------------
print_subheader("9C. Full Cross-Tab: first_break × ORB position — lift over base rate")
print(f"\n  Lift = observed% / base_rate%.  Values >1 mean over-representation.\n")

hi_sub = fb[fb['first_break_dir'] == 'high']
lo_sub = fb[fb['first_break_dir'] == 'low']

for orb_label, base_buck, label in [
    ('orb5_pos_bucket',  base_buck5,  'ORB-5'),
    ('orb15_pos_bucket', base_buck15, 'ORB-15'),
]:
    print(f"\n  {label}")
    print(f"  {'':25}  {'HIGH break%':>12}  {'LOW break%':>12}  {'Lift(HIGH)':>11}  {'Lift(LOW)':>10}")
    print(f"  {'-'*25}  {'-'*12}  {'-'*12}  {'-'*11}  {'-'*10}")
    for bucket in buckets:
        hi_pct  = safe_pct((hi_sub[orb_label] == bucket).sum(), len(hi_sub))
        lo_pct  = safe_pct((lo_sub[orb_label] == bucket).sum(), len(lo_sub))
        base    = base_buck[bucket]
        hi_lift = round(hi_pct / base, 2) if base > 0 else 0
        lo_lift = round(lo_pct / base, 2) if base > 0 else 0
        print(f"  {bucket:<25}  {hi_pct:>10.1f}%  {lo_pct:>10.1f}%  {hi_lift:>10.2f}x  {lo_lift:>9.2f}x")


# ------------------------------------------------------------------
# 9D. Conditioning on IB close half sharpens the picture
# ------------------------------------------------------------------
print_subheader("9D. First Break + IB Close Half → ORB-5 Position")
print(f"\n  When we know both the break direction AND the IB close half,")
print(f"  does the ORB position concentrate even more tightly?\n")

for break_dir in ['high', 'low']:
    # Theory-consistent IB close: high break → IB closed upper; low break → closed lower
    expected_close = 'upper' if break_dir == 'high' else 'lower'
    subset = fb[(fb['first_break_dir'] == break_dir) &
                (fb['ib_close_half']   == expected_close)]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE:
        continue
    print(f"  first_break={break_dir.upper()} + ib_close={expected_close}  (n={n})")
    print_reverse_table(subset, base_buck5, 'orb5_pos_bucket')
    print()

print_subheader("9E. First Break + IB Close Half → ORB-15 Position")

for break_dir in ['high', 'low']:
    expected_close = 'upper' if break_dir == 'high' else 'lower'
    subset = fb[(fb['first_break_dir'] == break_dir) &
                (fb['ib_close_half']   == expected_close)]
    n = len(subset)
    if n < MIN_DISPLAY_SAMPLE:
        continue
    print(f"\n  first_break={break_dir.upper()} + ib_close={expected_close}  (n={n})")
    print_reverse_table(subset, base_buck15, 'orb15_pos_bucket')


# ------------------------------------------------------------------
# 9F. Adding ORB close half to the reverse picture
# ------------------------------------------------------------------
print_subheader("9F. First Break × ORB-5 Close Half → ORB-5 Position")
print(f"\n  Given first_break=HIGH, how does ORB-5 position split by orb5_close_half?\n")

hi_sub_fb = fb[fb['first_break_dir'] == 'high']
for orb_close in ['upper', 'lower']:
    sub = hi_sub_fb[hi_sub_fb['orb5_close_half'] == orb_close]
    n = len(sub)
    if n < MIN_DISPLAY_SAMPLE: continue
    print(f"  first_break=HIGH + orb5_close={orb_close}  (n={n})")
    for bucket in buckets:
        count = (sub['orb5_pos_bucket'] == bucket).sum()
        pct   = safe_pct(count, n)
        bar   = pct_bar(pct, 30)
        print(f"    {bucket:<20}  {count:>5}  {pct:>5.1f}%  {bar}")
    print()

print(f"  Given first_break=LOW, how does ORB-5 position split by orb5_close_half?\n")
lo_sub_fb = fb[fb['first_break_dir'] == 'low']
for orb_close in ['upper', 'lower']:
    sub = lo_sub_fb[lo_sub_fb['orb5_close_half'] == orb_close]
    n = len(sub)
    if n < MIN_DISPLAY_SAMPLE: continue
    print(f"  first_break=LOW + orb5_close={orb_close}  (n={n})")
    for bucket in buckets:
        count = (sub['orb5_pos_bucket'] == bucket).sum()
        pct   = safe_pct(count, n)
        bar   = pct_bar(pct, 30)
        print(f"    {bucket:<20}  {count:>5}  {pct:>5.1f}%  {bar}")
    print()


# ------------------------------------------------------------------
# 9G. Predictive power summary — forward vs reverse comparison
# ------------------------------------------------------------------
print_subheader("9G. Predictive Power Summary — Forward vs Reverse")
print(f"""
  For the two strongest theoretically-aligned combinations:
    ORB-5  bottom_third → first break HIGH  (theory: ORB at bottom → market goes up → high broken first)
    ORB-5  top_third    → first break LOW
    ORB-15 bottom_third → first break HIGH
    ORB-15 top_third    → first break LOW
""")

for orb_label, base_buck, orb_name in [
    ('orb5_pos_bucket',  base_buck5,  'ORB-5'),
    ('orb15_pos_bucket', base_buck15, 'ORB-15'),
]:
    for bucket, expected_break in [('bottom_third', 'high'), ('top_third', 'low')]:

        fwd_sub  = fb[fb[orb_label] == bucket]
        fwd_n    = len(fwd_sub)
        fwd_hit  = (fwd_sub['first_break_dir'] == expected_break).sum()
        fwd_pct  = safe_pct(fwd_hit, fwd_n)

        rev_sub  = fb[fb['first_break_dir'] == expected_break]
        rev_n    = len(rev_sub)
        rev_hit  = (rev_sub[orb_label] == bucket).sum()
        rev_pct  = safe_pct(rev_hit, rev_n)

        base_fwd = base_hi_fb if expected_break == 'high' else base_lo_fb
        lift     = round(rev_pct / base_buck[bucket], 2) if base_buck[bucket] > 0 else 0

        print(f"  {orb_name} {bucket} → {expected_break} break")
        print(f"    Forward : {fwd_pct:.1f}% of {bucket} days → {expected_break} breaks first  (n={fwd_n})")
        print(f"    Reverse : {rev_pct:.1f}% of {expected_break}-break days had ORB in {bucket}  (n={rev_n})")
        print(f"    Base    : {base_buck[bucket]:.1f}% of all days have ORB in {bucket}")
        print(f"    Lift    : {lift:.2f}x  (reverse rate / base rate)")
        print()


# ============================================================
# SECTION 10 — SESSION CONTEXT → POST-IB OUTCOMES
# ============================================================
#
# Asia session   : 6:00pm–2:00am ET (prior day open to current day)
# London session : 2:00am–8:00am ET
#
# Session pattern : asia_dir + london_dir (e.g. 'bull_bull', 'bear_bear')
# Asia/London pos : structural range relationship between the two sessions
#
# ============================================================

print_header("SECTION 10 — SESSION CONTEXT → POST-IB OUTCOMES")

# Shared baseline for this section
sess_valid     = rdf[rdf['post_ib_direction'].notna()]
sess_base_bull = safe_pct((sess_valid['post_ib_direction'] == 'bullish').sum(), len(sess_valid))
fb_valid       = rdf[rdf['first_break_dir'].isin(['high', 'low'])]
sess_base_fb   = safe_pct((fb_valid['first_break_dir'] == 'high').sum(), len(fb_valid))

print(f"\n  Post-IB close baseline : Bull {sess_base_bull:.1f}%  (n={len(sess_valid)})")
print(f"  First break baseline   : High {sess_base_fb:.1f}%  (n={len(fb_valid)})\n")


def outcomes_dual(subset, label, base_bull, base_fb_hi, indent=4):
    """Print post-IB close direction AND first break direction for a subset."""
    # Close direction
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    total = bull + bear
    if total >= MIN_DISPLAY_SAMPLE:
        bull_pct  = safe_pct(bull, total)
        edge      = bull_pct - base_bull
        edge_flag = f"{edge:+.1f}%"
        bias      = "BULL ▲" if bull_pct > 50 else "BEAR ▼"
        print(f"{' '*indent}{label:<58} n={total:4d}  "
              f"Bull:{bull_pct:5.1f}%  Edge:{edge_flag:>7}  [{bias}]  close_dir")
    # First break
    hi = (subset['first_break_dir'] == 'high').sum()
    lo = (subset['first_break_dir'] == 'low').sum()
    total_fb = hi + lo
    if total_fb >= MIN_DISPLAY_SAMPLE:
        hi_pct    = safe_pct(hi, total_fb)
        edge_fb   = hi_pct - base_fb_hi
        edge_flag = f"{edge_fb:+.1f}%"
        bias_fb   = "HIGH ▲" if hi_pct > 50 else "LOW  ▼"
        print(f"{' '*indent}{label:<58} n={total_fb:4d}  "
              f"High:{hi_pct:5.1f}%  Edge:{edge_flag:>7}  [{bias_fb}]  first_break")


print_subheader("Session Pattern (Asia + London combined direction)")
for pat in ['bull_bull', 'bull_bear', 'bear_bull', 'bear_bear']:
    sub = rdf[rdf['session_pattern'] == pat]
    n = len(sub)
    print(f"\n  session_pattern = {pat}  (n={n})")
    outcomes_dual(sub, pat, sess_base_bull, sess_base_fb)

print_subheader("Asia / London Positional Relationship")
positions = ['london_partial_up', 'london_partial_down', 'london_breaks_both',
             'london_engulfs_asia', 'asia_engulfs_london', 'overlapping']
for pos in positions:
    sub = rdf[rdf['asia_london_pos'] == pos]
    outcomes_dual(sub, pos, sess_base_bull, sess_base_fb)

print_subheader("Session Pattern × IB Close Half (2-factor)")
for pat in ['bull_bull', 'bear_bear']:
    for ch in ['upper', 'lower']:
        sub = rdf[(rdf['session_pattern'] == pat) & (rdf['ib_close_half'] == ch)]
        outcomes_dual(sub, f"{pat} + ib_close={ch}", sess_base_bull, sess_base_fb)

print_subheader("Session Pattern × ORB-5 Position (2-factor)")
for pat in ['bull_bull', 'bear_bear']:
    for bucket in ['bottom_third', 'top_third']:
        sub = rdf[(rdf['session_pattern'] == pat) & (rdf['orb5_pos_bucket'] == bucket)]
        outcomes_dual(sub, f"{pat} + orb5_pos={bucket}", sess_base_bull, sess_base_fb)

print_subheader("Asia/London Position × IB Close Half (2-factor)")
for pos in ['london_partial_up', 'london_partial_down']:
    for ch in ['upper', 'lower']:
        sub = rdf[(rdf['asia_london_pos'] == pos) & (rdf['ib_close_half'] == ch)]
        outcomes_dual(sub, f"{pos} + ib_close={ch}", sess_base_bull, sess_base_fb)

print_subheader("Session Pattern × Asia/London Position × IB Close Half (3-factor)")
for pat in ['bull_bull', 'bear_bear']:
    for pos in ['london_partial_up', 'london_partial_down']:
        for ch in ['upper', 'lower']:
            sub = rdf[(rdf['session_pattern'] == pat) &
                      (rdf['asia_london_pos']  == pos) &
                      (rdf['ib_close_half']    == ch)]
            outcomes_dual(sub, f"{pat}+{pos}+ib={ch}", sess_base_bull, sess_base_fb)


# ============================================================
# SECTION 11 — SIGNAL SCORING SYSTEM
# ============================================================
#
#   Signal 1 : session_pattern     (Asia + London combined direction)
#   Signal 2 : asia_london_pos     (structural relationship)
#   Signal 3 : orb5_close_half     (where ORB-5 closed in its own range)
#   Signal 4 : orb5_pos_bucket     (where ORB-5 sits in full IB range)
#   Signal 5 : ib_extreme_first    (which IB extreme printed first)
#   Signal 6 : ib_close_half       (where IB closed in its own range)
#
#   Score ≥ 4/6 → directional bias outputted
#   Score < 4/6 → NO EDGE / STAND ASIDE
#
#   Trade type:
#     EXTENSION : low_first + IB_close_upper  → extension long
#                 high_first + IB_close_lower → extension short
#     FADE      : high_first + IB_close_upper → fade short (go long)
#                 low_first + IB_close_lower  → fade long (go short)
#
# ============================================================

print_header("SECTION 11 — SIGNAL SCORING SYSTEM (4/6 MINIMUM)")

base_bull_s11  = sess_base_bull
base_fb_s11    = sess_base_fb

print_subheader("Score Label Distribution")
for label in ['bullish', 'bearish', 'no_edge']:
    n   = (rdf['score_label'] == label).sum()
    pct = safe_pct(n, total_days)
    bar = "█" * int(pct / 2)
    print(f"    {label:<12} : {n:5d}  ({pct:5.1f}%)  {bar}")

print_subheader("Scored Bullish (≥4/6 bull signals) → Outcomes")
sub = rdf[rdf['score_label'] == 'bullish']
outcomes_dual(sub, "score=bullish", base_bull_s11, base_fb_s11)

print_subheader("Scored Bearish (≥4/6 bear signals) → Outcomes")
sub = rdf[rdf['score_label'] == 'bearish']
outcomes_dual(sub, "score=bearish", base_bull_s11, base_fb_s11)

print_subheader("No Edge / Stand Aside (<4 signals agree) → Outcomes")
sub = rdf[rdf['score_label'] == 'no_edge']
outcomes_dual(sub, "score=no_edge", base_bull_s11, base_fb_s11)

print_subheader("Trade Type Distribution")
for tt in ['extension_long', 'extension_short', 'fade_long', 'fade_short', 'unclear']:
    n   = (rdf['trade_type'] == tt).sum()
    pct = safe_pct(n, total_days)
    bar = "█" * int(pct / 2)
    print(f"    {tt:<20} : {n:5d}  ({pct:5.1f}%)  {bar}")

print_subheader("Trade Type → Outcomes")
for tt in ['extension_long', 'extension_short', 'fade_long', 'fade_short']:
    sub = rdf[rdf['trade_type'] == tt]
    outcomes_dual(sub, f"trade_type={tt}", base_bull_s11, base_fb_s11)

print_subheader("Score Label × Trade Type → Outcomes")
for score in ['bullish', 'bearish']:
    aligned_tt = 'extension_long' if score == 'bullish' else 'extension_short'
    fade_tt    = 'fade_long'      if score == 'bullish' else 'fade_short'
    for tt in [aligned_tt, fade_tt]:
        sub = rdf[(rdf['score_label'] == score) & (rdf['trade_type'] == tt)]
        outcomes_dual(sub, f"score={score} + trade={tt}", base_bull_s11, base_fb_s11)

print_subheader("Score Label × Session Pattern → Outcomes")
for score in ['bullish', 'bearish']:
    aligned_pat = 'bull_bull' if score == 'bullish' else 'bear_bear'
    sub = rdf[(rdf['score_label'] == score) & (rdf['session_pattern'] == aligned_pat)]
    outcomes_dual(sub, f"score={score} + session={aligned_pat}", base_bull_s11, base_fb_s11)

print_subheader("Score Label × IB Close Half → Outcomes")
for score in ['bullish', 'bearish']:
    aligned_close = 'upper' if score == 'bullish' else 'lower'
    sub = rdf[(rdf['score_label'] == score) & (rdf['ib_close_half'] == aligned_close)]
    outcomes_dual(sub, f"score={score} + ib_close={aligned_close}", base_bull_s11, base_fb_s11)

print_subheader("Full Ranked Edge Table — All Patterns Including Session + Signals")
print(f"\n  Baseline: Bull {base_bull_s11:.1f}%  |  Min sample: {MIN_DISPLAY_SAMPLE}\n")

all_edge_rows = []

def collect_all(label, subset):
    bull  = (subset['post_ib_direction'] == 'bullish').sum()
    bear  = (subset['post_ib_direction'] == 'bearish').sum()
    n     = bull + bear
    if n < MIN_DISPLAY_SAMPLE: return
    bull_pct = safe_pct(bull, n)
    edge     = abs(bull_pct - base_bull_s11)
    bias     = 'BULL' if bull_pct > 50 else 'BEAR'
    all_edge_rows.append({'label': label, 'n': n,
                          'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
                          'edge': edge, 'bias': bias})

# Session single factors
for pat in ['bull_bull', 'bull_bear', 'bear_bull', 'bear_bear']:
    collect_all(f"session_pattern={pat}", rdf[rdf['session_pattern'] == pat])
for pos in positions:
    collect_all(f"asia_lon_pos={pos}", rdf[rdf['asia_london_pos'] == pos])

# Signal system
for sl in ['bullish', 'bearish', 'no_edge']:
    collect_all(f"score_label={sl}", rdf[rdf['score_label'] == sl])
for tt in ['extension_long', 'extension_short', 'fade_long', 'fade_short']:
    collect_all(f"trade_type={tt}", rdf[rdf['trade_type'] == tt])

# ORB + IB single factors
for val in ['upper', 'lower']:
    collect_all(f"orb5_close_half={val}",  rdf[rdf['orb5_close_half']  == val])
    collect_all(f"orb15_close_half={val}", rdf[rdf['orb15_close_half'] == val])
    collect_all(f"ib_close_half={val}",    rdf[rdf['ib_close_half']    == val])
for b in ['bottom_third', 'middle_third', 'top_third']:
    collect_all(f"orb5_pos={b}",  rdf[rdf['orb5_pos_bucket']  == b])
    collect_all(f"orb15_pos={b}", rdf[rdf['orb15_pos_bucket'] == b])
for ext in ['high', 'low']:
    collect_all(f"ib_extreme_first={ext}", rdf[rdf['ib_extreme_first'] == ext])
for ct in ['high_first', 'low_first', 'both_first_half', 'both_second_half']:
    collect_all(f"ib_curve_type={ct}", rdf[rdf['ib_curve_type'] == ct])

# 2-factor combos
for ext in ['high', 'low']:
    for ch in ['upper', 'lower']:
        collect_all(f"extreme={ext}+ib_close={ch}",
                    rdf[(rdf['ib_extreme_first']==ext) & (rdf['ib_close_half']==ch)])
for b in ['bottom_third', 'top_third']:
    for ch in ['upper', 'lower']:
        collect_all(f"orb5_pos={b}+ib_close={ch}",
                    rdf[(rdf['orb5_pos_bucket']==b) & (rdf['ib_close_half']==ch)])
        collect_all(f"orb15_pos={b}+ib_close={ch}",
                    rdf[(rdf['orb15_pos_bucket']==b) & (rdf['ib_close_half']==ch)])
for pat in ['bull_bull', 'bear_bear']:
    for ch in ['upper', 'lower']:
        collect_all(f"session={pat}+ib_close={ch}",
                    rdf[(rdf['session_pattern']==pat) & (rdf['ib_close_half']==ch)])
    for b in ['bottom_third', 'top_third']:
        collect_all(f"session={pat}+orb5_pos={b}",
                    rdf[(rdf['session_pattern']==pat) & (rdf['orb5_pos_bucket']==b)])
for sl in ['bullish', 'bearish']:
    for tt in ['extension_long', 'extension_short', 'fade_long', 'fade_short']:
        collect_all(f"score={sl}+trade={tt}",
                    rdf[(rdf['score_label']==sl) & (rdf['trade_type']==tt)])
    for ch in ['upper', 'lower']:
        collect_all(f"score={sl}+ib_close={ch}",
                    rdf[(rdf['score_label']==sl) & (rdf['ib_close_half']==ch)])

# 3-factor combos
for pat in ['bull_bull', 'bear_bear']:
    for ch in ['upper', 'lower']:
        for b in ['bottom_third', 'top_third']:
            collect_all(f"session={pat}+ib={ch}+orb5={b}",
                        rdf[(rdf['session_pattern']==pat) &
                            (rdf['ib_close_half']==ch) &
                            (rdf['orb5_pos_bucket']==b)])
for orb_ch in ['upper', 'lower']:
    for b in ['bottom_third', 'top_third']:
        for ib_ch in ['upper', 'lower']:
            collect_all(f"orb5_close={orb_ch}+{b}+ib={ib_ch}",
                        meaningful5[(meaningful5['orb5_close_half']==orb_ch) &
                                    (meaningful5['orb5_pos_bucket']==b) &
                                    (meaningful5['ib_close_half']==ib_ch)])
            collect_all(f"orb15_close={orb_ch}+{b}+ib={ib_ch}",
                        meaningful15[(meaningful15['orb15_close_half']==orb_ch) &
                                     (meaningful15['orb15_pos_bucket']==b) &
                                     (meaningful15['ib_close_half']==ib_ch)])

all_edge_df = pd.DataFrame(all_edge_rows).sort_values('edge', ascending=False)

print(f"  {'PATTERN':<60} {'n':>6}  {'BULL%':>6}  {'BEAR%':>6}  {'EDGE':>6}  {'BIAS':>5}")
print(f"  {'-'*60}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*5}")
for _, row in all_edge_df.iterrows():
    print(f"  {row['label']:<60} {row['n']:>6}  "
          f"{row['bull_pct']:>5.1f}%  {row['bear_pct']:>5.1f}%  "
          f"{row['edge']:>5.1f}%  {row['bias']:>5}")

# Save ranked edge table
all_edge_df.to_csv(EDGE_CSV_PATH, index=False)
print(f"\n  Ranked edge table saved : {EDGE_CSV_PATH}")

print_subheader("How to Use This Script Live")
print(f"""
  After 10:30am each day, you have all 6 signals:

  ┌─────────────────────────────────────────────────────────────────────────┐
  │  Signal 1 : session_pattern   — What did Asia + London do together?     │
  │  Signal 2 : asia_london_pos   — How do Asia/London ranges relate?       │
  │  Signal 3 : orb5_close_half   — Where did the ORB-5 close in its range? │
  │  Signal 4 : orb5_pos_bucket   — Where did the ORB-5 sit in the IB?      │
  │  Signal 5 : ib_extreme_first  — Which IB extreme printed first?         │
  │  Signal 6 : ib_close_half     — Where did the IB close in its range?    │
  └─────────────────────────────────────────────────────────────────────────┘

  Score each signal bullish / bearish / neutral.
  ≥ {MIN_SIGNAL_AGREEMENT}/6 signals agree → directional bias with edge.
  < {MIN_SIGNAL_AGREEMENT}/6 signals agree → NO EDGE / STAND ASIDE.

  Then check trade_type:
    EXTENSION : market moved one way and held → trade the continuation
    FADE      : market tried one direction, rejected → trade the other side

  Outputs:
    Dataset CSV  : {DATASET_CSV_PATH}
    Edge table   : {EDGE_CSV_PATH}
""")


# ============================================================
# ANALYSIS COMPLETE
# ============================================================

print_header("ANALYSIS COMPLETE")
print(f"""
  Chains investigated:
    Asia/London context
        → orb5_close_half  → orb5_pos_in_ib  → ib_extreme_first → ib_close_half → post-IB
        → orb15_close_half → orb15_pos_in_ib → ib_extreme_first → ib_close_half → post-IB

  Key questions answered:
    1.  How does ORB-15 internal structure compare to ORB-5?
    2.  Does ORB-15 close half predict IB position better than ORB-5?
    3.  Do ORB-5 and ORB-15 agree on close half, and does agreement matter?
    4.  Does ORB-15 position in the IB carry more or less edge than ORB-5?
    5.  ORB-15 curve (Q2 breaks Q1): how does continuation rate compare to ORB-5?
    6.  When both ORB-5 and ORB-15 confirm the same curve/continuation, is the
        post-IB edge meaningfully stronger?
    7.  Which 3-factor confluence (ORB-5 vs ORB-15 version) has the highest edge?
    8.  Can the ORB (5-min and/or 15-min) predict which IB extreme forms first?
        — See Section 2C for the ranked accuracy summary by signal and timing.
    9.  Reverse conditional: given a post-IB break direction, what was the ORB setup?
        — Section 9 gives forward accuracy, reverse diagnostic rates, and lift scores.
    10. Does Asia/London session context add directional edge to the IB framework?
        — Section 10 tests session pattern, positional relationship, and 3-factor combos.
    11. Does a 6-signal majority vote system produce meaningful edge above baseline?
        — Section 11: score distribution, trade type, ranked confluence edge table.

  Outputs saved:
    Full dataset  : {DATASET_CSV_PATH}
    Edge table    : {EDGE_CSV_PATH}
""")


                  UNIFIED IB / ORB CURVE ANALYSIS  (5-min + 15-min ORB)                   

  Loading data from: C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
  Loaded 4,161,727 bars | 3786 trading days (2013-01-01 → 2025-12-31)

  Building per-day records...
    Day 500 of 3785...
    Day 1000 of 3785...
    Day 1500 of 3785...
    Day 2000 of 3785...
    Day 2500 of 3785...
    Day 3000 of 3785...
    Day 3500 of 3785...

  Records built : 3,124  |  Skipped: 661
  Valid post-IB : 3,116
  Dataset saved : C:\Users\tammm\NQ and ES analysis\ib_opportunity_dataset.csv

                    SECTION 1A — ORB-5 INTERNAL STRUCTURE (9:30–9:35)                     
  Where does the 5-min ORB close within its own range?
  (Only meaningful ORBs shown — range ≥ 3% of prev day range)

  Total days              : 3,124
  ORB-5 range meaningful  : 2,480  (79.4%)
  ORB-5 range too tight   : 644  (20.6%)

------------------------------------------------------------------------------------------
  O

In [1]:
# ============================================================
# UNIFIED IB / ORB CURVE ANALYSIS  — v4
# (+ Session Context, Signal Scoring, London/Asia Taxonomy,
#  NY Open vs London/Overnight Midpoint)
# ============================================================
#
# Investigates the full auction-theory chain for NQ futures:
#
#   Asia/London context (4-way taxonomy)
#       → ny_open_vs_london_mid  / ny_open_vs_overnight_mid
#           → orb_close_half  (5-min and 15-min)
#               → orb_position_in_ib
#                   → ib_extreme_first
#                       → ib_close_half
#                           → post-IB direction (10:30am → 4:00pm)
#
# CHANGES FROM v3
# ---------------
#   • Asia/London taxonomy unified with scripts 2 & 3:
#       london_engulfs_asia          (was london_breaks_both)
#       asia_engulfs_london          (was asia_engulfs_london — same)
#       london_partially_engulfs_upwards   (was london_partial_up)
#       london_partially_engulfs_downwards (was london_partial_down)
#     The old 'overlapping' category is eliminated; the four above
#     are exhaustive and mutually exclusive by construction.
#
#   • Two new computed fields per day:
#       ny_open_vs_london_mid    — 'above' / 'below'
#       ny_open_vs_overnight_mid — 'above' / 'below'
#
#   • Signal 7 : ny_open_vs_london_mid
#     Signal 8 : ny_open_vs_overnight_mid
#     (scoring system extended from 6 to 8 signals;
#      MIN_SIGNAL_AGREEMENT raised from 4 to 5 by default)
#
#   • Section 12 (new):
#       12A. London/Asia Pattern × IB Signals → Post-IB
#       12B. NY Open vs London Midpoint → Post-IB / First Break
#       12C. NY Open vs Overnight Midpoint → Post-IB / First Break
#       12D. 3-factor: Asia/London Pattern × NY Open Position × IB Close Half
#       12E. 4-factor confluence table (highest-edge patterns)
#
# SECTIONS
# --------
#   1A–1B.  ORB Internal Structure (5-min / 15-min)
#   2A–2C.  ORB Position Within IB / IB Extreme Predictor
#   3.      IB Quarter Structure
#   4A–4B.  ORB Curve Analysis (5-min / 15-min)
#   5.      Q1 Curve Analysis
#   6.      Chain Validation
#   7.      Post-IB Session Outcomes
#   8.      Full Confluence Summary
#   9.      Reverse Conditional Analysis
#   10.     Session Context → Post-IB Outcomes
#   11.     Signal Scoring System (8-signal)
#   12.     London/Asia Taxonomy + NY Open Midpoint Analysis (NEW)
#
# ============================================================

import pandas as pd
import numpy as np
import warnings
from collections import defaultdict
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR      = r"C:\Users\tammm\NQ and ES analysis"
RAW_DATA_FILE = "nq_15y_1min.csv"
OUTPUT_DIR    = r"C:\Users\tammm\NQ and ES analysis"
START_DATE    = "2013-01-01"
END_DATE      = "2025-12-31"

MIN_ORB_RANGE_FRACTION = 0.03   # 3% of prev day range
MIN_SIGNAL_AGREEMENT   = 5      # raised from 4 → 5 (now 8 signals total)
MIN_DISPLAY_SAMPLE     = 30

import os
RAW_DATA_PATH    = os.path.join(DATA_DIR, RAW_DATA_FILE)
DATASET_CSV_PATH = os.path.join(OUTPUT_DIR, "ib_opportunity_dataset_v4.csv")
EDGE_CSV_PATH    = os.path.join(OUTPUT_DIR, "ib_opportunity_edge_table_v4.csv")


# ============================================================
# HELPERS
# ============================================================

def safe_pct(numerator, denominator):
    return (numerator / denominator * 100) if denominator > 0 else 0.0


def close_half(close_price, range_high, range_low):
    if range_high == range_low:
        return None
    midpoint = (range_high + range_low) / 2
    return 'upper' if close_price >= midpoint else 'lower'


def position_in_range(price, range_high, range_low):
    if range_high == range_low:
        return None
    return (price - range_low) / (range_high - range_low)


def position_bucket(score):
    if score is None:
        return None
    if score <= 1/3:
        return 'bottom_third'
    elif score <= 2/3:
        return 'middle_third'
    else:
        return 'top_third'


def session_direction(open_price, close_price):
    if open_price == close_price:
        return None
    return 'bullish' if close_price > open_price else 'bearish'


def classify_asia_london(london_high, london_low, asia_high, asia_low):
    """
    Four-way exhaustive taxonomy (matches scripts 2 & 3):
      london_engulfs_asia          : London high > Asia high AND London low < Asia low
      asia_engulfs_london          : London high <= Asia high AND London low >= Asia low
      london_partially_engulfs_upwards  : London high > Asia high AND London low >= Asia low
      london_partially_engulfs_downwards: London low < Asia low AND London high <= Asia high
    """
    if london_high > asia_high and london_low < asia_low:
        return 'london_engulfs_asia'
    elif london_high <= asia_high and london_low >= asia_low:
        return 'asia_engulfs_london'
    elif london_high > asia_high and london_low >= asia_low:
        return 'london_partially_engulfs_upwards'
    elif london_low < asia_low and london_high <= asia_high:
        return 'london_partially_engulfs_downwards'
    else:
        # Mathematically unreachable with the four conditions above,
        # but kept as a safety net for floating-point edge cases.
        return 'other'


def print_header(text, width=90):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=90):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar_len = int(pct / 2)
    bar = "█" * bar_len
    print(f"{' '*indent}{label:<55} {count:5d}  ({pct:5.1f}%)  {bar}")


def print_outcome_row(label, bull, bear, indent=4):
    total = bull + bear
    if total < MIN_DISPLAY_SAMPLE:
        print(f"{' '*indent}{label:<55} n={total:4d}  (below min sample)")
        return
    bull_pct = safe_pct(bull, total)
    bear_pct = safe_pct(bear, total)
    edge = abs(bull_pct - 50)
    bias = "BULL" if bull_pct > 50 else "BEAR"
    print(f"{' '*indent}{label:<55} n={total:4d}  "
          f"Bull:{bull_pct:5.1f}%  Bear:{bear_pct:5.1f}%  "
          f"Edge:{edge:4.1f}%  [{bias}]")


# ============================================================
# SIGNAL SCORING
# ============================================================

# Updated Asia/London bias map using 4-way taxonomy
ASIA_LONDON_BIAS = {
    'london_engulfs_asia'               : 'neutral',
    'asia_engulfs_london'               : 'neutral',
    'london_partially_engulfs_upwards'  : 'bullish',
    'london_partially_engulfs_downwards': 'bearish',
}

NY_OPEN_VS_MID_BIAS = {
    'above': 'bullish',
    'below': 'bearish',
}

def score_asia_london_pattern(pattern):
    return ASIA_LONDON_BIAS.get(pattern, 'neutral')

def score_orb_close_half(orb_close_h, orb_meaningful):
    if not orb_meaningful or orb_close_h is None: return 'neutral'
    return 'bullish' if orb_close_h == 'upper' else 'bearish'

def score_orb_position(bucket):
    if bucket == 'bottom_third': return 'bullish'
    if bucket == 'top_third':    return 'bearish'
    return 'neutral'

def score_ib_extreme_first(extreme):
    if extreme == 'low':  return 'bullish'
    if extreme == 'high': return 'bearish'
    return 'neutral'

def score_ib_close_half(close_h):
    if close_h == 'upper': return 'bullish'
    if close_h == 'lower': return 'bearish'
    return 'neutral'

def score_ny_open_vs_mid(position):
    """'above' London/overnight midpoint → bullish; 'below' → bearish."""
    return NY_OPEN_VS_MID_BIAS.get(position, 'neutral')

def compute_trade_type(ib_extreme, ib_close_h):
    if ib_extreme is None or ib_close_h is None: return 'unclear'
    if ib_extreme == 'low'  and ib_close_h == 'upper': return 'extension_long'
    if ib_extreme == 'high' and ib_close_h == 'lower': return 'extension_short'
    if ib_extreme == 'high' and ib_close_h == 'upper': return 'fade_long'
    if ib_extreme == 'low'  and ib_close_h == 'lower': return 'fade_short'
    return 'unclear'

def aggregate_signals(scores, min_agreement=5):
    bull = scores.count('bullish')
    bear = scores.count('bearish')
    neut = scores.count('neutral')
    agreement_met = max(bull, bear) >= min_agreement
    if bull == 0 and bear == 0:
        return 'no_edge', bull, bear, neut, False
    bias = 'bullish' if bull >= bear else 'bearish'
    return bias, bull, bear, neut, agreement_met


# ============================================================
# LOAD DATA
# ============================================================

print_header("UNIFIED IB / ORB CURVE ANALYSIS  v4  (8-signal system)")
print(f"\n  Loading data from: {RAW_DATA_PATH}")

df = pd.read_csv(RAW_DATA_PATH)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df = df.loc[START_DATE:END_DATE]

df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

print(f"  Loaded {len(df):,} bars | {len(dates)} trading days ({START_DATE} → {END_DATE})")


# ============================================================
# CURVE LABEL MAP
# ============================================================

curve_labels = {
    'broke_high_curved_low'     : '{ref} HIGH → 2nd half curves LOW   [CURVE]',
    'broke_high_continued_high' : '{ref} HIGH → 2nd half extends HIGH  [CONTINUATION]',
    'broke_high_neither'        : '{ref} HIGH → 2nd half stays in range',
    'broke_low_curved_high'     : '{ref} LOW  → 2nd half curves HIGH   [CURVE]',
    'broke_low_continued_low'   : '{ref} LOW  → 2nd half extends LOW   [CONTINUATION]',
    'broke_low_neither'         : '{ref} LOW  → 2nd half stays in range',
    'q2_breaks_both'            : 'Q2 breaks BOTH {ref} extremes',
    'q2_contained'              : 'Q2 contained within {ref} range',
}

def make_curve_label(result, ref):
    return curve_labels[result].replace('{ref}', f'Q2 breaks {ref}')


# ============================================================
# MAIN LOOP
# ============================================================

print("\n  Building per-day records...")

records = []
skipped = 0

for idx, date in enumerate(dates[1:], 1):

    if idx % 500 == 0:
        print(f"    Day {idx} of {len(dates)-1}...")

    actual = date + pd.Timedelta(hours=7)

    # ── Time boundaries ──────────────────────────────────────────────────────

    asia_start  = actual.replace(hour=20) - pd.Timedelta(days=1)  # 8 PM prior day
    asia_end    = actual.replace(hour=2)                           # 2 AM
    lon_start   = actual.replace(hour=2)
    lon_end     = actual.replace(hour=8)

    orb5_start  = actual.replace(hour=9,  minute=30)
    orb5_end    = actual.replace(hour=9,  minute=35)
    ib_start    = actual.replace(hour=9,  minute=30)
    q1_end      = actual.replace(hour=9,  minute=45)
    q2_end      = actual.replace(hour=10, minute=0)
    q3_end      = actual.replace(hour=10, minute=15)
    ib_end      = actual.replace(hour=10, minute=30)
    ny_end      = actual.replace(hour=16, minute=0)

    # ── Slice data ───────────────────────────────────────────────────────────

    asia_data    = df.loc[asia_start:asia_end]
    lon_data     = df.loc[lon_start:lon_end]
    orb5_data    = df.loc[orb5_start:orb5_end]
    orb15_data   = df.loc[ib_start:q1_end]
    q1_data      = df.loc[ib_start:q1_end]
    q2_data      = df.loc[q1_end:q2_end]
    q3_data      = df.loc[q2_end:q3_end]
    q4_data      = df.loc[q3_end:ib_end]
    ib_data      = df.loc[ib_start:ib_end]
    post_ib_data = df.loc[ib_end:ny_end]

    if any(d.empty for d in [asia_data, lon_data, orb5_data, orb15_data,
                              q1_data, q2_data, ib_data, post_ib_data]):
        skipped += 1; continue
    if len(orb5_data) < 5:
        skipped += 1; continue

    # ── Previous day range ───────────────────────────────────────────────────

    prev_actual = (date - pd.Timedelta(days=1)) + pd.Timedelta(hours=7)
    prev_data   = df.loc[
        prev_actual.replace(hour=18) - pd.Timedelta(days=1)
        : prev_actual.replace(hour=17)
    ]
    prev_range = (prev_data['High'].max() - prev_data['Low'].min()) if not prev_data.empty else None

    # ── ORB-5 ────────────────────────────────────────────────────────────────

    orb5_open  = orb5_data['Open'].iloc[0]
    orb5_close = orb5_data['Close'].iloc[-1]
    orb5_high  = orb5_data['High'].max()
    orb5_low   = orb5_data['Low'].min()
    orb5_range = orb5_high - orb5_low

    if pd.isna([orb5_open, orb5_close, orb5_high, orb5_low]).any():
        skipped += 1; continue

    orb5_range_meaningful = (
        prev_range is not None and prev_range > 0 and
        orb5_range >= prev_range * MIN_ORB_RANGE_FRACTION
    )
    orb5_close_half  = close_half(orb5_close, orb5_high, orb5_low) if orb5_range_meaningful else None
    orb5_direction   = session_direction(orb5_open, orb5_close)
    orb5_mid         = (orb5_high + orb5_low) / 2

    # ── ORB-15 ───────────────────────────────────────────────────────────────

    orb15_open  = orb15_data['Open'].iloc[0]
    orb15_close = orb15_data['Close'].iloc[-1]
    orb15_high  = orb15_data['High'].max()
    orb15_low   = orb15_data['Low'].min()
    orb15_range = orb15_high - orb15_low

    if pd.isna([orb15_open, orb15_close, orb15_high, orb15_low]).any():
        skipped += 1; continue

    orb15_range_meaningful = (
        prev_range is not None and prev_range > 0 and
        orb15_range >= prev_range * MIN_ORB_RANGE_FRACTION
    )
    orb15_close_half = close_half(orb15_close, orb15_high, orb15_low) if orb15_range_meaningful else None
    orb15_direction  = session_direction(orb15_open, orb15_close)
    orb15_mid        = (orb15_high + orb15_low) / 2

    # ── IB metrics ───────────────────────────────────────────────────────────

    ib_open  = ib_data['Open'].iloc[0]
    ib_close = ib_data['Close'].iloc[-1]
    ib_high  = ib_data['High'].max()
    ib_low   = ib_data['Low'].min()
    ib_range = ib_high - ib_low

    if pd.isna([ib_open, ib_close, ib_high, ib_low]).any() or ib_range == 0:
        skipped += 1; continue

    ib_close_half_val = close_half(ib_close, ib_high, ib_low)

    # ── ORB position within IB ───────────────────────────────────────────────

    orb5_pos_in_ib   = position_in_range(orb5_mid,  ib_high, ib_low)
    orb5_pos_bucket  = position_bucket(orb5_pos_in_ib)
    orb15_pos_in_ib  = position_in_range(orb15_mid, ib_high, ib_low)
    orb15_pos_bucket = position_bucket(orb15_pos_in_ib)

    # ── IB extreme first ─────────────────────────────────────────────────────

    ib_extreme_first_val = None
    for _, bar in ib_data.iterrows():
        hit_high = bar['High'] >= ib_high
        hit_low  = bar['Low']  <= ib_low
        if hit_high and hit_low:
            ib_extreme_first_val = 'simultaneous'; break
        elif hit_high:
            ib_extreme_first_val = 'high'; break
        elif hit_low:
            ib_extreme_first_val = 'low'; break

    # ── IB quarter structure ─────────────────────────────────────────────────

    def which_quarter(timestamp):
        if timestamp < q1_end:   return 1
        if timestamp < q2_end:   return 2
        if timestamp < q3_end:   return 3
        return 4

    ib_high_time = ib_data[ib_data['High'] == ib_high].index[0]
    ib_low_time  = ib_data[ib_data['Low']  == ib_low].index[0]
    ib_high_q    = which_quarter(ib_high_time)
    ib_low_q     = which_quarter(ib_low_time)
    ib_high_half = 1 if ib_high_time < q2_end else 2
    ib_low_half  = 1 if ib_low_time  < q2_end else 2

    if ib_high_half != ib_low_half:
        ib_curve_type = 'high_first' if ib_high_half == 1 else 'low_first'
    elif ib_high_half == 1:
        ib_curve_type = 'both_first_half'
    else:
        ib_curve_type = 'both_second_half'

    # ── Q1 / Q2 / second-half ranges ─────────────────────────────────────────

    q1_high = q1_data['High'].max()
    q1_low  = q1_data['Low'].min()
    q2_high = q2_data['High'].max()
    q2_low  = q2_data['Low'].min()

    second_half_data = df.loc[q2_end:ib_end]
    if second_half_data.empty:
        skipped += 1; continue

    sh_high = second_half_data['High'].max()
    sh_low  = second_half_data['Low'].min()

    # ── ORB-5 curve ───────────────────────────────────────────────────────────

    q2_breaks_orb5_high = q2_high > orb5_high
    q2_breaks_orb5_low  = q2_low  < orb5_low

    orb5_curve_result = None
    if q2_breaks_orb5_high and not q2_breaks_orb5_low:
        if sh_low < orb5_low:
            orb5_curve_result = 'broke_high_curved_low'
        elif sh_high > q2_high:
            orb5_curve_result = 'broke_high_continued_high'
        else:
            orb5_curve_result = 'broke_high_neither'
    elif q2_breaks_orb5_low and not q2_breaks_orb5_high:
        if sh_high > orb5_high:
            orb5_curve_result = 'broke_low_curved_high'
        elif sh_low < q2_low:
            orb5_curve_result = 'broke_low_continued_low'
        else:
            orb5_curve_result = 'broke_low_neither'
    elif q2_breaks_orb5_high and q2_breaks_orb5_low:
        orb5_curve_result = 'q2_breaks_both'
    else:
        orb5_curve_result = 'q2_contained'

    # ── ORB-15 curve ──────────────────────────────────────────────────────────

    q2_breaks_orb15_high = q2_high > orb15_high
    q2_breaks_orb15_low  = q2_low  < orb15_low

    orb15_curve_result = None
    if q2_breaks_orb15_high and not q2_breaks_orb15_low:
        if sh_low < orb15_low:
            orb15_curve_result = 'broke_high_curved_low'
        elif sh_high > q2_high:
            orb15_curve_result = 'broke_high_continued_high'
        else:
            orb15_curve_result = 'broke_high_neither'
    elif q2_breaks_orb15_low and not q2_breaks_orb15_high:
        if sh_high > orb15_high:
            orb15_curve_result = 'broke_low_curved_high'
        elif sh_low < q2_low:
            orb15_curve_result = 'broke_low_continued_low'
        else:
            orb15_curve_result = 'broke_low_neither'
    elif q2_breaks_orb15_high and q2_breaks_orb15_low:
        orb15_curve_result = 'q2_breaks_both'
    else:
        orb15_curve_result = 'q2_contained'

    # ── Q1 curve (Section 5) ──────────────────────────────────────────────────

    q2_breaks_q1_high = q2_high > q1_high
    q2_breaks_q1_low  = q2_low  < q1_low

    q1_curve_result = None
    if q2_breaks_q1_high and not q2_breaks_q1_low:
        if sh_low < q1_low:
            q1_curve_result = 'broke_high_curved_low'
        elif sh_high > q2_high:
            q1_curve_result = 'broke_high_continued_high'
        else:
            q1_curve_result = 'broke_high_neither'
    elif q2_breaks_q1_low and not q2_breaks_q1_high:
        if sh_high > q1_high:
            q1_curve_result = 'broke_low_curved_high'
        elif sh_low < q2_low:
            q1_curve_result = 'broke_low_continued_low'
        else:
            q1_curve_result = 'broke_low_neither'
    elif q2_breaks_q1_high and q2_breaks_q1_low:
        q1_curve_result = 'q2_breaks_both'
    else:
        q1_curve_result = 'q2_contained'

    # ── Post-IB direction ─────────────────────────────────────────────────────

    post_ib_close = post_ib_data['Close'].iloc[-1]
    post_ib_dir   = session_direction(ib_close, post_ib_close)

    # ── First IB extreme broken post-IB ──────────────────────────────────────

    first_break_dir = None
    for _, bar in post_ib_data.iterrows():
        broke_high = bar['High'] > ib_high
        broke_low  = bar['Low']  < ib_low
        if broke_high and broke_low:
            first_break_dir = 'simultaneous'; break
        elif broke_high:
            first_break_dir = 'high'; break
        elif broke_low:
            first_break_dir = 'low'; break

    # ── Chain agreement flags ─────────────────────────────────────────────────

    orb5_ib_close_agree  = (orb5_close_half  is not None and
                             ib_close_half_val is not None and
                             orb5_close_half  == ib_close_half_val)
    orb15_ib_close_agree = (orb15_close_half is not None and
                             ib_close_half_val is not None and
                             orb15_close_half == ib_close_half_val)

    extreme_close_agree = None
    if ib_extreme_first_val in ('high', 'low') and ib_close_half_val is not None:
        expected_close = 'lower' if ib_extreme_first_val == 'high' else 'upper'
        extreme_close_agree = (ib_close_half_val == expected_close)

    full_confluence_5 = None
    if orb5_close_half and orb5_pos_bucket and ib_close_half_val:
        bull5 = (orb5_close_half == 'upper' and orb5_pos_bucket == 'bottom_third' and ib_close_half_val == 'upper')
        bear5 = (orb5_close_half == 'lower' and orb5_pos_bucket == 'top_third'    and ib_close_half_val == 'lower')
        full_confluence_5 = 'bullish' if bull5 else ('bearish' if bear5 else 'mixed')

    full_confluence_15 = None
    if orb15_close_half and orb15_pos_bucket and ib_close_half_val:
        bull15 = (orb15_close_half == 'upper' and orb15_pos_bucket == 'bottom_third' and ib_close_half_val == 'upper')
        bear15 = (orb15_close_half == 'lower' and orb15_pos_bucket == 'top_third'    and ib_close_half_val == 'lower')
        full_confluence_15 = 'bullish' if bull15 else ('bearish' if bear15 else 'mixed')

    # ── Asia / London session metrics ─────────────────────────────────────────

    asia_open  = asia_data['Open'].iloc[0]
    asia_close = asia_data['Close'].iloc[-1]
    asia_high  = asia_data['High'].max()
    asia_low   = asia_data['Low'].min()
    asia_dir   = session_direction(asia_open, asia_close)

    lon_open  = lon_data['Open'].iloc[0]
    lon_close = lon_data['Close'].iloc[-1]
    lon_high  = lon_data['High'].max()
    lon_low   = lon_data['Low'].min()
    lon_dir   = session_direction(lon_open, lon_close)

    if pd.isna([asia_high, asia_low, lon_high, lon_low]).any():
        skipped += 1; continue

    # ── NEW: 4-way Asia/London taxonomy ──────────────────────────────────────

    asia_london_pattern = classify_asia_london(lon_high, lon_low, asia_high, asia_low)

    # Session direction combo (for reference / Section 10)
    if asia_dir and lon_dir:
        session_dir_pattern = f"{asia_dir[:4]}_{lon_dir[:4]}"
    else:
        session_dir_pattern = None

    # ── NEW: NY open vs London midpoint ──────────────────────────────────────

    # NY opens at 9:30 RTH, but we use the 8 AM futures open as in scripts 2/3
    # To be consistent with scripts 2/3 which use ny_data.Open (8 AM open):
    ny_8am_data = df.loc[lon_end:lon_end + pd.Timedelta(minutes=1)]
    ny_8am_open = ny_8am_data['Open'].iloc[0] if not ny_8am_data.empty else None

    if ny_8am_open is None and not post_ib_data.empty:
        ny_8am_open = ib_data['Open'].iloc[0]  # fallback to IB open

    lon_mid = (lon_high + lon_low) / 2
    overnight_high = max(asia_high, lon_high)
    overnight_low  = min(asia_low,  lon_low)
    overnight_mid  = (overnight_high + overnight_low) / 2

    ny_open_vs_london_mid    = None
    ny_open_vs_overnight_mid = None

    if ny_8am_open is not None:
        ny_open_vs_london_mid    = 'above' if ny_8am_open > lon_mid    else 'below'
        ny_open_vs_overnight_mid = 'above' if ny_8am_open > overnight_mid else 'below'

    # ── 8-signal scoring ─────────────────────────────────────────────────────

    # Signal 1 : Asia direction (bull/bear)
    sig1 = 'bullish' if asia_dir == 'bullish' else ('bearish' if asia_dir == 'bearish' else 'neutral')
    # Signal 2 : London direction
    sig2 = 'bullish' if lon_dir  == 'bullish' else ('bearish' if lon_dir  == 'bearish' else 'neutral')
    # Signal 3 : Asia/London structural pattern
    sig3 = score_asia_london_pattern(asia_london_pattern)
    # Signal 4 : ORB-5 close half
    sig4 = score_orb_close_half(orb5_close_half, orb5_range_meaningful)
    # Signal 5 : ORB-5 position in IB
    sig5 = score_orb_position(orb5_pos_bucket)
    # Signal 6 : IB extreme first
    sig6 = score_ib_extreme_first(ib_extreme_first_val)
    # Signal 7 : IB close half
    sig7 = score_ib_close_half(ib_close_half_val)
    # Signal 8 : NY 8am open vs London midpoint
    sig8 = score_ny_open_vs_mid(ny_open_vs_london_mid)

    signals = [sig1, sig2, sig3, sig4, sig5, sig6, sig7, sig8]
    bias, bull_count, bear_count, neut_count, agreement_met = aggregate_signals(
        signals, MIN_SIGNAL_AGREEMENT)
    score_label = bias if agreement_met else 'no_edge'

    trade_type = compute_trade_type(ib_extreme_first_val, ib_close_half_val)

    records.append({
        # Identifiers
        'date'                      : actual.date(),
        'prev_range'                : prev_range,

        # ── Session context ──
        'asia_direction'            : asia_dir,
        'london_direction'          : lon_dir,
        'session_dir_pattern'       : session_dir_pattern,
        'asia_london_pattern'       : asia_london_pattern,       # NEW 4-way
        'ny_open_vs_london_mid'     : ny_open_vs_london_mid,     # NEW
        'ny_open_vs_overnight_mid'  : ny_open_vs_overnight_mid,  # NEW

        # ── Signal scores ──
        'sig1_asia_dir'             : sig1,
        'sig2_london_dir'           : sig2,
        'sig3_pattern'              : sig3,
        'sig4_orb_close_half'       : sig4,
        'sig5_orb_position'         : sig5,
        'sig6_ib_extreme_first'     : sig6,
        'sig7_ib_close_half'        : sig7,
        'sig8_ny_vs_lon_mid'        : sig8,
        'bull_signal_count'         : bull_count,
        'bear_signal_count'         : bear_count,
        'neut_signal_count'         : neut_count,
        'score_label'               : score_label,
        'agreement_met'             : agreement_met,
        'trade_type'                : trade_type,

        # ── ORB-5 ──
        'orb5_range'                : orb5_range,
        'orb5_range_meaningful'     : orb5_range_meaningful,
        'orb5_direction'            : orb5_direction,
        'orb5_close_half'           : orb5_close_half,
        'orb5_pos_in_ib'            : orb5_pos_in_ib,
        'orb5_pos_bucket'           : orb5_pos_bucket,
        'orb5_curve_result'         : orb5_curve_result,
        'orb5_ib_close_agree'       : orb5_ib_close_agree,
        'full_confluence_5'         : full_confluence_5,

        # ── ORB-15 ──
        'orb15_range'               : orb15_range,
        'orb15_range_meaningful'    : orb15_range_meaningful,
        'orb15_direction'           : orb15_direction,
        'orb15_close_half'          : orb15_close_half,
        'orb15_pos_in_ib'           : orb15_pos_in_ib,
        'orb15_pos_bucket'          : orb15_pos_bucket,
        'orb15_curve_result'        : orb15_curve_result,
        'orb15_ib_close_agree'      : orb15_ib_close_agree,
        'full_confluence_15'        : full_confluence_15,

        # ── IB structure ──
        'ib_close_half'             : ib_close_half_val,
        'ib_extreme_first'          : ib_extreme_first_val,
        'ib_curve_type'             : ib_curve_type,
        'ib_high_quarter'           : ib_high_q,
        'ib_low_quarter'            : ib_low_q,
        'ib_high_half'              : ib_high_half,
        'ib_low_half'               : ib_low_half,

        # ── Q1 curve ──
        'q1_curve_result'           : q1_curve_result,

        # ── Chain flags ──
        'extreme_close_agree'       : extreme_close_agree,

        # ── Outcomes ──
        'post_ib_direction'         : post_ib_dir,
        'first_break_dir'           : first_break_dir,
    })

rdf = pd.DataFrame(records)
total_days    = len(rdf)
valid_post_ib = rdf['post_ib_direction'].notna().sum()

print(f"\n  Records built : {total_days:,}  |  Skipped: {skipped}")
print(f"  Valid post-IB : {valid_post_ib:,}")

rdf.to_csv(DATASET_CSV_PATH, index=False)
print(f"  Dataset saved : {DATASET_CSV_PATH}")

meaningful5  = rdf[rdf['orb5_range_meaningful']  == True]
meaningful15 = rdf[rdf['orb15_range_meaningful'] == True]

# Baselines used throughout
baseline_bull  = (rdf['post_ib_direction'] == 'bullish').sum()
baseline_bear  = (rdf['post_ib_direction'] == 'bearish').sum()
baseline_total = baseline_bull + baseline_bear
base_bull_pct  = safe_pct(baseline_bull, baseline_total)
fb_valid       = rdf[rdf['first_break_dir'].isin(['high', 'low'])]
base_fb_hi     = safe_pct((fb_valid['first_break_dir'] == 'high').sum(), len(fb_valid))


# ============================================================
# SECTIONS 1A–11 (UNCHANGED FROM v3 — abbreviated here)
# Full implementations are identical to v3; only the new
# fields and Section 12 are reproduced in full below.
# ============================================================
# NOTE: Paste your v3 Sections 1A through 11 here verbatim.
# The record schema is backward-compatible: every field from
# v3 exists in v4 under the same name. The only schema change
# is that 'asia_london_pos' is renamed 'asia_london_pattern'
# and uses the 4-way taxonomy. Update any Section 10
# references from 'asia_london_pos' to 'asia_london_pattern'.
# ============================================================


# ============================================================
# SECTION 12 — LONDON/ASIA TAXONOMY + NY OPEN MIDPOINT
# ============================================================

print_header("SECTION 12 — LONDON/ASIA TAXONOMY + NY OPEN MIDPOINT ANALYSIS")
print(f"""
  This section integrates two new context layers into the IB framework:

    1. Asia/London structural pattern (4-way exhaustive taxonomy):
         london_engulfs_asia             : London range fully contains Asia range
         asia_engulfs_london             : Asia range fully contains London range
         london_partially_engulfs_upwards: London extends above Asia but not below
         london_partially_engulfs_downwards: London extends below Asia but not above

    2. NY 8am open position relative to:
         (a) London session midpoint     [ny_open_vs_london_mid]
         (b) Overnight (Asia+London) midpoint [ny_open_vs_overnight_mid]

  All outcomes measured as post-IB direction (10:30am → 4:00pm close).
  Baseline: Bull {base_bull_pct:.1f}%  (n={baseline_total})
""")

# ── Shared helper for this section ───────────────────────────────────────────

def outcomes_dual_12(subset, label, indent=2):
    """Print post-IB close AND first-break edge for a subset."""
    sp = ' ' * indent
    # Close direction
    bull = (subset['post_ib_direction'] == 'bullish').sum()
    bear = (subset['post_ib_direction'] == 'bearish').sum()
    total = bull + bear
    if total >= MIN_DISPLAY_SAMPLE:
        bull_pct  = safe_pct(bull, total)
        edge      = bull_pct - base_bull_pct
        flag      = f"{edge:+.1f}%"
        bias_tag  = "BULL ▲" if bull_pct > 50 else "BEAR ▼"
        print(f"{sp}{label:<60} n={total:4d}  "
              f"Bull:{bull_pct:5.1f}%  Edge:{flag:>7}  [{bias_tag}]  close_dir")
    # First break direction
    hi = (subset['first_break_dir'] == 'high').sum()
    lo = (subset['first_break_dir'] == 'low').sum()
    total_fb = hi + lo
    if total_fb >= MIN_DISPLAY_SAMPLE:
        hi_pct    = safe_pct(hi, total_fb)
        edge_fb   = hi_pct - base_fb_hi
        flag_fb   = f"{edge_fb:+.1f}%"
        bias_fb   = "HIGH ▲" if hi_pct > 50 else "LOW  ▼"
        print(f"{sp}{label:<60} n={total_fb:4d}  "
              f"High:{hi_pct:5.1f}%  Edge:{flag_fb:>7}  [{bias_fb}]  first_break")


# ────────────────────────────────────────────────────────────────────────────
# 12A.  Asia/London Pattern — occurrence and raw post-IB edge
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12A. Asia/London Pattern: Occurrence and Post-IB Edge")
print(f"  (4-way exhaustive taxonomy; compares with v3 6-way to show coverage)\n")

PATTERNS_4WAY = [
    'london_engulfs_asia',
    'asia_engulfs_london',
    'london_partially_engulfs_upwards',
    'london_partially_engulfs_downwards',
]

print(f"  {'PATTERN':<44}  {'n':>6}  {'%days':>7}")
print(f"  {'-'*44}  {'-'*6}  {'-'*7}")
for pat in PATTERNS_4WAY + ['other']:
    n = (rdf['asia_london_pattern'] == pat).sum()
    print(f"  {pat:<44}  {n:>6}  ({safe_pct(n, total_days):>5.1f}%)")

print()
for pat in PATTERNS_4WAY:
    sub = rdf[rdf['asia_london_pattern'] == pat]
    outcomes_dual_12(sub, pat)

# ────────────────────────────────────────────────────────────────────────────
# 12B.  NY Open vs London Midpoint
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12B. NY 8am Open vs London Midpoint → Post-IB Edge")
print(f"  Theory: NY opens above London midpoint → bullish auction bias\n")

for pos in ['above', 'below']:
    sub = rdf[rdf['ny_open_vs_london_mid'] == pos]
    outcomes_dual_12(sub, f"NY 8am opens {pos} London midpoint")

print()
# Broken down by Asia/London pattern
print(f"  --- By Asia/London Pattern ---\n")
for pat in PATTERNS_4WAY:
    for pos in ['above', 'below']:
        sub = rdf[(rdf['asia_london_pattern'] == pat) &
                  (rdf['ny_open_vs_london_mid'] == pos)]
        outcomes_dual_12(sub, f"{pat[:36]} + NY {pos} lon_mid", indent=4)
    print()

# ────────────────────────────────────────────────────────────────────────────
# 12C.  NY Open vs Overnight Midpoint
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12C. NY 8am Open vs Overnight Midpoint → Post-IB Edge")
print(f"  Overnight = max(Asia, London) high/low combined range\n")

for pos in ['above', 'below']:
    sub = rdf[rdf['ny_open_vs_overnight_mid'] == pos]
    outcomes_dual_12(sub, f"NY 8am opens {pos} overnight midpoint")

print()
print(f"  --- Overnight mid vs London mid: agreement / split ---\n")
both_above = rdf[(rdf['ny_open_vs_london_mid'] == 'above') &
                 (rdf['ny_open_vs_overnight_mid'] == 'above')]
both_below = rdf[(rdf['ny_open_vs_london_mid'] == 'below') &
                 (rdf['ny_open_vs_overnight_mid'] == 'below')]
split_lo_on = rdf[(rdf['ny_open_vs_london_mid'] == 'above') &
                  (rdf['ny_open_vs_overnight_mid'] == 'below')]
split_on_lo = rdf[(rdf['ny_open_vs_london_mid'] == 'below') &
                  (rdf['ny_open_vs_overnight_mid'] == 'above')]

outcomes_dual_12(both_above,  "Both above (London & overnight)", indent=4)
outcomes_dual_12(both_below,  "Both below (London & overnight)", indent=4)
outcomes_dual_12(split_lo_on, "Above London mid BUT below overnight mid", indent=4)
outcomes_dual_12(split_on_lo, "Below London mid BUT above overnight mid", indent=4)

# ────────────────────────────────────────────────────────────────────────────
# 12D.  Asia/London Pattern × NY Open Position × IB Close Half
#        3-factor conditioning
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12D. Asia/London Pattern × NY Open vs London Mid × IB Close Half")
print(f"  3-factor conditioning — strongest pre-IB + IB confluence\n")

for pat in PATTERNS_4WAY:
    pat_sub = rdf[rdf['asia_london_pattern'] == pat]
    n_pat = len(pat_sub)
    if n_pat < MIN_DISPLAY_SAMPLE:
        continue
    print(f"\n  ── {pat.upper()} (n={n_pat}) ──")

    for ny_pos in ['above', 'below']:
        for ib_ch in ['upper', 'lower']:
            sub = pat_sub[(pat_sub['ny_open_vs_london_mid'] == ny_pos) &
                          (pat_sub['ib_close_half']         == ib_ch)]
            outcomes_dual_12(sub,
                             f"  NY {ny_pos} lon_mid + IB_close={ib_ch}",
                             indent=6)

# ────────────────────────────────────────────────────────────────────────────
# 12E.  Asia/London Pattern × NY Open Position × IB Extreme First × IB Close Half
#        4-factor — highest granularity
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12E. 4-Factor: Pattern × NY Open × IB Extreme First × IB Close Half")
print(f"  Only cells with n ≥ {MIN_DISPLAY_SAMPLE} shown. Theory-aligned cells flagged.\n")

edge_4f_rows = []

for pat in PATTERNS_4WAY:
    for ny_pos in ['above', 'below']:
        for extreme in ['high', 'low']:
            for ib_ch in ['upper', 'lower']:
                sub = rdf[
                    (rdf['asia_london_pattern']    == pat)     &
                    (rdf['ny_open_vs_london_mid']  == ny_pos)  &
                    (rdf['ib_extreme_first']        == extreme) &
                    (rdf['ib_close_half']           == ib_ch)
                ]
                bull  = (sub['post_ib_direction'] == 'bullish').sum()
                bear  = (sub['post_ib_direction'] == 'bearish').sum()
                total = bull + bear
                if total < MIN_DISPLAY_SAMPLE:
                    continue
                bull_pct = safe_pct(bull, total)
                edge     = abs(bull_pct - base_bull_pct)
                bias     = 'BULL' if bull_pct > 50 else 'BEAR'
                # Theory alignment: pattern bull + NY above + low_first + upper → extension long
                theory_bull = (pat == 'london_partially_engulfs_upwards' and
                               ny_pos == 'above' and extreme == 'low' and ib_ch == 'upper')
                theory_bear = (pat == 'london_partially_engulfs_downwards' and
                               ny_pos == 'below' and extreme == 'high' and ib_ch == 'lower')
                tag = '★BULL' if theory_bull else ('★BEAR' if theory_bear else '')
                edge_4f_rows.append({
                    'pattern'  : pat,
                    'ny_pos'   : ny_pos,
                    'extreme'  : extreme,
                    'ib_close' : ib_ch,
                    'n'        : total,
                    'bull_pct' : bull_pct,
                    'edge'     : edge,
                    'bias'     : bias,
                    'tag'      : tag,
                })

edge_4f_df = pd.DataFrame(edge_4f_rows).sort_values('edge', ascending=False)

print(f"  {'PATTERN':<38} {'NY':>5}  {'EXT':>5}  {'IB':>5}  {'n':>5}  "
      f"{'BULL%':>6}  {'EDGE':>5}  {'BIAS':>5}  {'NOTE':>6}")
print(f"  {'-'*38}  {'-'*5}  {'-'*5}  {'-'*5}  {'-'*5}  "
      f"{'-'*6}  {'-'*5}  {'-'*5}  {'-'*6}")

for _, row in edge_4f_df.iterrows():
    print(f"  {row['pattern']:<38} {row['ny_pos']:>5}  {row['extreme']:>5}  "
          f"{row['ib_close']:>5}  {row['n']:>5}  "
          f"{row['bull_pct']:>5.1f}%  {row['edge']:>5.1f}%  "
          f"{row['bias']:>5}  {row['tag']:>6}")

# ────────────────────────────────────────────────────────────────────────────
# 12F.  Asia/London Pattern × ORB-5 Close Half × IB Close Half
#        Cross-validates the ORB signal within each structural pattern
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12F. Asia/London Pattern × ORB-5 Close Half × IB Close Half")
print(f"  Does the ORB-5 signal carry different edge in each structural pattern?\n")

for pat in PATTERNS_4WAY:
    pat_sub = meaningful5[meaningful5['asia_london_pattern'] == pat]
    n_pat = len(pat_sub)
    if n_pat < MIN_DISPLAY_SAMPLE:
        continue
    print(f"\n  ── {pat} (n={n_pat}) ──")
    for orb_ch in ['upper', 'lower']:
        for ib_ch in ['upper', 'lower']:
            sub = pat_sub[(pat_sub['orb5_close_half'] == orb_ch) &
                          (pat_sub['ib_close_half']   == ib_ch)]
            outcomes_dual_12(sub,
                             f"  ORB-5={orb_ch} + IB_close={ib_ch}",
                             indent=6)

# ────────────────────────────────────────────────────────────────────────────
# 12G.  NY Open vs London Mid × ORB-5 Close Half × IB Close Half
#        3-factor ORB confluence conditioned on NY open position
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12G. NY Open vs London Mid × ORB-5 Close Half × IB Close Half")
print(f"  Does knowing the NY open position sharpen the ORB-5 signal?\n")

for ny_pos in ['above', 'below']:
    ny_sub = meaningful5[meaningful5['ny_open_vs_london_mid'] == ny_pos]
    n_ny = len(ny_sub)
    print(f"\n  ── NY opens {ny_pos} London mid (n={n_ny}) ──")
    for orb_ch in ['upper', 'lower']:
        for ib_ch in ['upper', 'lower']:
            sub = ny_sub[(ny_sub['orb5_close_half'] == orb_ch) &
                         (ny_sub['ib_close_half']   == ib_ch)]
            outcomes_dual_12(sub,
                             f"  ORB-5={orb_ch} + IB_close={ib_ch}",
                             indent=6)

# ────────────────────────────────────────────────────────────────────────────
# 12H.  Ranked edge table — all new patterns (Section 12 only)
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12H. Section 12 Ranked Edge Table — all patterns sorted by edge")
print(f"\n  Baseline: Bull {base_bull_pct:.1f}%  |  Min sample: {MIN_DISPLAY_SAMPLE}\n")

edge_12_rows = []

def collect_12(label, subset):
    bull  = (subset['post_ib_direction'] == 'bullish').sum()
    bear  = (subset['post_ib_direction'] == 'bearish').sum()
    n     = bull + bear
    if n < MIN_DISPLAY_SAMPLE: return
    bull_pct = safe_pct(bull, n)
    edge     = abs(bull_pct - base_bull_pct)
    bias     = 'BULL' if bull_pct > 50 else 'BEAR'
    edge_12_rows.append({'label': label, 'n': n,
                         'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
                         'edge': edge, 'bias': bias})

# Single factors
for pat in PATTERNS_4WAY:
    collect_12(f"asia_london={pat}", rdf[rdf['asia_london_pattern'] == pat])
for pos in ['above', 'below']:
    collect_12(f"ny_vs_lon_mid={pos}",   rdf[rdf['ny_open_vs_london_mid']    == pos])
    collect_12(f"ny_vs_on_mid={pos}",    rdf[rdf['ny_open_vs_overnight_mid'] == pos])

# 2-factor
for pat in PATTERNS_4WAY:
    for pos in ['above', 'below']:
        collect_12(f"pattern={pat[:28]}+ny={pos}",
                   rdf[(rdf['asia_london_pattern']   == pat) &
                       (rdf['ny_open_vs_london_mid'] == pos)])
    for ib_ch in ['upper', 'lower']:
        collect_12(f"pattern={pat[:28]}+ib={ib_ch}",
                   rdf[(rdf['asia_london_pattern'] == pat) &
                       (rdf['ib_close_half']        == ib_ch)])

for pos in ['above', 'below']:
    for ib_ch in ['upper', 'lower']:
        collect_12(f"ny={pos}+ib={ib_ch}",
                   rdf[(rdf['ny_open_vs_london_mid'] == pos) &
                       (rdf['ib_close_half']          == ib_ch)])
    for ext in ['high', 'low']:
        collect_12(f"ny={pos}+extreme={ext}",
                   rdf[(rdf['ny_open_vs_london_mid'] == pos) &
                       (rdf['ib_extreme_first']       == ext)])
    for orb_ch in ['upper', 'lower']:
        collect_12(f"ny={pos}+orb5={orb_ch}",
                   meaningful5[(meaningful5['ny_open_vs_london_mid'] == pos) &
                                (meaningful5['orb5_close_half']       == orb_ch)])

# 3-factor
for pat in PATTERNS_4WAY:
    for pos in ['above', 'below']:
        for ib_ch in ['upper', 'lower']:
            collect_12(f"pat={pat[:22]}+ny={pos}+ib={ib_ch}",
                       rdf[(rdf['asia_london_pattern']   == pat) &
                           (rdf['ny_open_vs_london_mid'] == pos) &
                           (rdf['ib_close_half']          == ib_ch)])

for pos in ['above', 'below']:
    for orb_ch in ['upper', 'lower']:
        for ib_ch in ['upper', 'lower']:
            collect_12(f"ny={pos}+orb5={orb_ch}+ib={ib_ch}",
                       meaningful5[(meaningful5['ny_open_vs_london_mid'] == pos) &
                                   (meaningful5['orb5_close_half']        == orb_ch) &
                                   (meaningful5['ib_close_half']           == ib_ch)])

edge_12_df = pd.DataFrame(edge_12_rows).sort_values('edge', ascending=False)

print(f"  {'PATTERN':<62} {'n':>6}  {'BULL%':>6}  {'BEAR%':>6}  {'EDGE':>6}  {'BIAS':>5}")
print(f"  {'-'*62}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*5}")
for _, row in edge_12_df.iterrows():
    print(f"  {row['label']:<62} {row['n']:>6}  "
          f"{row['bull_pct']:>5.1f}%  {row['bear_pct']:>5.1f}%  "
          f"{row['edge']:>5.1f}%  {row['bias']:>5}")

# ────────────────────────────────────────────────────────────────────────────
# 12I.  8-Signal System: distribution and outcomes
#        (replaces v3 Section 11 signal summary for the new 8-signal system)
# ────────────────────────────────────────────────────────────────────────────

print_subheader("12I. 8-Signal System: Distribution and Outcomes")
print(f"""
  Signal 1 : asia_direction          — Asia session bullish/bearish
  Signal 2 : london_direction        — London session bullish/bearish
  Signal 3 : asia_london_pattern     — structural pattern (partial up/down = directional)
  Signal 4 : orb5_close_half         — ORB-5 closes upper/lower half of its range
  Signal 5 : orb5_pos_bucket         — ORB-5 midpoint in bottom/top third of IB
  Signal 6 : ib_extreme_first        — which IB extreme formed first
  Signal 7 : ib_close_half           — IB closes upper/lower half of its range
  Signal 8 : ny_open_vs_london_mid   — NY 8am opens above/below London midpoint

  Agreement threshold : ≥ {MIN_SIGNAL_AGREEMENT}/8 signals → directional bias
""")

print_subheader("Score Label Distribution (8-signal, threshold ≥ 5)")
for label in ['bullish', 'bearish', 'no_edge']:
    n   = (rdf['score_label'] == label).sum()
    pct = safe_pct(n, total_days)
    bar = "█" * int(pct / 2)
    print(f"    {label:<12} : {n:5d}  ({pct:5.1f}%)  {bar}")

print_subheader("Scored Bullish → Outcomes")
outcomes_dual_12(rdf[rdf['score_label'] == 'bullish'], "score=bullish")

print_subheader("Scored Bearish → Outcomes")
outcomes_dual_12(rdf[rdf['score_label'] == 'bearish'], "score=bearish")

print_subheader("No Edge / Stand Aside → Outcomes")
outcomes_dual_12(rdf[rdf['score_label'] == 'no_edge'], "score=no_edge")

print_subheader("Trade Type × Score Label → Outcomes")
for score in ['bullish', 'bearish']:
    for tt in ['extension_long', 'extension_short', 'fade_long', 'fade_short']:
        sub = rdf[(rdf['score_label'] == score) & (rdf['trade_type'] == tt)]
        outcomes_dual_12(sub, f"score={score} + trade={tt}", indent=4)

# ────────────────────────────────────────────────────────────────────────────
# Save Section 12 edge table and finish
# ────────────────────────────────────────────────────────────────────────────

edge_12_df.to_csv(EDGE_CSV_PATH, index=False)
print(f"\n  Section 12 edge table saved : {EDGE_CSV_PATH}")

print_header("ANALYSIS COMPLETE — v4")
print(f"""
  New in v4 vs v3:
    • Asia/London taxonomy unified to 4-way exhaustive classification
    • NY 8am open vs London midpoint added as Signal 8
    • NY 8am open vs overnight midpoint computed (diagnostic)
    • Scoring system extended to 8 signals (agreement threshold: ≥ {MIN_SIGNAL_AGREEMENT})
    • Section 12 (A–I): full cross-tabulation of structural pattern,
      NY open position, ORB signals, and IB signals

  Outputs:
    Full dataset  : {DATASET_CSV_PATH}
    Edge table    : {EDGE_CSV_PATH}

  Usage reminder after 10:30am each day:
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  Sig 1 : Asia direction (bull/bear)                                     │
  │  Sig 2 : London direction (bull/bear)                                   │
  │  Sig 3 : Asia/London structural pattern → partial_up=bull, partial_down │
  │  Sig 4 : ORB-5 closes upper/lower half of its range  [known at 9:35]   │
  │  Sig 5 : ORB-5 sits in bottom/top third of IB range  [known at 10:30]  │
  │  Sig 6 : IB extreme first (low=bull, high=bear)      [known at 10:30]  │
  │  Sig 7 : IB closes upper/lower half                  [known at 10:30]  │
  │  Sig 8 : NY 8am open above/below London midpoint     [known at 8:00]   │
  └─────────────────────────────────────────────────────────────────────────┘
  ≥ {MIN_SIGNAL_AGREEMENT}/8 agree → take the bias.  Trade type: extension vs fade per Sig 6 × Sig 7.
""")


                  UNIFIED IB / ORB CURVE ANALYSIS  v4  (8-signal system)                  

  Loading data from: C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
  Loaded 4,161,727 bars | 3786 trading days (2013-01-01 → 2025-12-31)

  Building per-day records...
    Day 500 of 3785...
    Day 1000 of 3785...
    Day 1500 of 3785...
    Day 2000 of 3785...
    Day 2500 of 3785...
    Day 3000 of 3785...
    Day 3500 of 3785...

  Records built : 3,124  |  Skipped: 661
  Valid post-IB : 3,116
  Dataset saved : C:\Users\tammm\NQ and ES analysis\ib_opportunity_dataset_v4.csv

              SECTION 12 — LONDON/ASIA TAXONOMY + NY OPEN MIDPOINT ANALYSIS               

  This section integrates two new context layers into the IB framework:

    1. Asia/London structural pattern (4-way exhaustive taxonomy):
         london_engulfs_asia             : London range fully contains Asia range
         asia_engulfs_london             : Asia range fully contains London range
         london_partial